## EDA, Sistema de Predicción Energética y Alerta Temprana en Barrios de Barcelona

Contexto: dataset con consumo eléctrico por código postal (bloques de 6h, 2019–2025), enriquecido con variables meteorológicas de estaciones Meteocat y temperatura superficial satelital (MODIS LST). El objetivo del EDA es comprender la estructura, calidad y comportamiento del dataset antes del modelado: identificar patrones temporales, relaciones entre variables y anomalías que condicionen las decisiones de feature engineering y arquitectura del modelo.

### <font color='#C0392B'><b>Variables del dataset</b></font>

| Variable | Tipo | Descripción |
|---|---|---|
| `cod_postal` | Categórica | Código postal de Barcelona (08001–08042) |
| `nombre_postal` | Categórica | Nombre del barrio asociado al código postal |
| `centroide_lon` | Numérica | Longitud del centroide del código postal |
| `centroide_lat` | Numérica | Latitud del centroide del código postal |
| `codi_estacio` | Categórica | Código de estación Meteocat asignada |
| `nombre_estacio` | Categórica | Nombre de la estación Meteocat asignada |
| `estacio_lon` | Numérica | Longitud de la estación Meteocat asignada |
| `estacio_lat` | Numérica | Latitud de la estación Meteocat asignada |
| `datetime` | Temporal | Inicio del bloque de 6 horas |
| `mwh_total` | Target | Consumo eléctrico total en MWh |
| `mwh_industria` | Numérica | Consumo sector industria |
| `mwh_residencial` | Numérica | Consumo sector residencial |
| `mwh_servicios` | Numérica | Consumo sector servicios |
| `mwh_no_especificado` | Numérica | Consumo no clasificado |
| `lst_celsius` | Numérica | Land Surface Temperature — MODIS (°C) |
| `temp_mean` | Numérica | Temperatura media Meteocat (°C) |
| `temp_max` | Numérica | Temperatura máxima Meteocat (°C) |
| `temp_min` | Numérica | Temperatura mínima Meteocat (°C) |
| `humedad_mean` | Numérica | Humedad relativa media (%) |
| `viento_mean` | Numérica | Velocidad media del viento (m/s) |
| `precipitacion_sum` | Numérica | Precipitación acumulada en el bloque (mm) |
| `irradiancia_mean` | Numérica | Irradiancia solar global media (W/m²) |
| `es_festivo` | Binaria | Es día festivo |
| `nombre_local` | Categórica | Nombre del festivo local |
| `hora` | Ordinal | Hora de inicio del bloque (0, 6, 12, 18) |
| `dia_semana` | Ordinal | Día de la semana (1=Lunes, 7=Domingo) |
| `mes` | Ordinal | Mes del año (1–12) |
| `anio` | Numérica | Año |
| `semana_anio` | Numérica | Semana del año (1–53) |
| `es_finde` | Binaria | Es fin de semana |

## <font color='#1B3A5C'>  **Librerías** </font>

In [ ]:
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import numpy as np
import scipy.stats as stats
from scipy.stats import kruskal, mannwhitneyu
from scipy.signal import correlate
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from pymongo import MongoClient
from statsmodels.tsa.seasonal import STL
from arch.unitroot import ADF, KPSS
from datetime import date
import json
import plotly.express as px
import pandas as pd
import time
from arch.unitroot import ADF, KPSS

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'
plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.xmargin'] = 0.01
plt.rcParams['axes.ymargin'] = 0.01

C1 = '#264653'
C2 = '#2A9D8F'
C3 = '#E9C46A'
C4 = '#F4A261'
C5 = '#E76F51'
C6 = '#A8DADC'
TITULO    = '#1B3A5C'
SUBTITULO = '#C0392B'

# <font color='#1B3A5C'>  **1. Carga y Exploración** </font>

### <font color='#C0392B'><b>1.1 Exploración Inicial</b></font>


In [ ]:
start_time = time.time()

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs = list(db['dataset_eda'].find({}, {'_id': 0}))
df   = pl.DataFrame(docs, infer_schema_length=None)

In [ ]:
df.head(6)

In [ ]:
df.tail(6)

In [ ]:
df.filter(pl.col("codi_estacio") == "X2").select(pl.col(["temp_mean", "temp_max", "temp_min", "humedad_mean", "viento_mean", "precipitacion_sum"])).head(6)

In [ ]:
print(f"Shape: {df.shape}")
print(f"Desde: {df['datetime'].min()}")
print(f"Hasta: {df['datetime'].max()}")
print(f"Códigos postales únicos: {df['cod_postal'].n_unique()}")
print(f"Años cubiertos: {sorted(df['anio'].unique().to_list())}")

In [ ]:
print("SCHEMA")
for col, dtype in zip(df.columns, df.dtypes):
    print(f"  {col:<30} {dtype}")

In [ ]:
pl.Config.set_tbl_rows(50)

nulls = (
    pl.DataFrame({
        'columna': df.columns,
        'nulos':   [df[c].null_count() for c in df.columns]
    })
    .with_columns((pl.col('nulos') / len(df) * 100).round(2).alias('pct_nulos'))
    .sort('nulos', descending=True)
    .filter(pl.col('nulos') > 0)
)

print(nulls)

El dataset cubre 424.148 registros en 42 códigos postales de Barcelona, en bloques de 6h desde enero de 2019 hasta junio de 2025. Cada fila representa un único bloque por código postal (datetime y cod_postal como clave compuesta) y agrupa variables de consumo eléctrico, meteorología Meteocat y temperatura superficial MODIS LST.

El análisis de nulos identifica tres grupos. El primero corresponde a nombre_local (96%) y mwh_no_especificado (85%): nombre_local es el nombre de la festividad, por lo que tiene sentido, y mwh_no_especificado es un valor añadido en 2025 al dataset. El segundo son las variables meteorológicas (32–36%). El tercero es la LST MODIS (39.8%), que refleja la cobertura nubosa y actúa como respaldo espacial. El resto deberá revisarse.

### <font color='#C0392B'><b>1.2 Distribución Inicial de datos faltantes</b></font>


In [ ]:
codpos_null = (
df.group_by('anio').agg([
    pl.col('irradiancia_mean').is_null().sum().alias('nulos_irradiancia'),
    pl.col('viento_mean').is_null().sum().alias('nulos_viento'),
    pl.col('temp_mean').is_null().sum().alias('nulos_temp'),
    pl.col('humedad_mean').is_null().sum().alias('nulos_humedad'),
    pl.len().alias('total_registros')
]).sort('anio'))

print(codpos_null)

In [ ]:
codpos_null = (df.filter(pl.col('viento_mean').is_null())
    .group_by(['cod_postal','nombre_postal','codi_estacio'])
    .agg(pl.len().alias('nulos_viento'))
    .sort('nulos_viento', descending=True)
    .select(['cod_postal','nombre_postal', 'nulos_viento','codi_estacio']))

print(codpos_null)

Los nulos de viento e irradiancia son constantes desde 2019: la estación X2 no dispone de estos sensores para la mayoría de códigos postales, lo que explica los 22.000 nulos constantes por año. Los códigos postales asignados a D5, X4 y X8 presentan nulos esporádicos, lo que confirma que el problema es exclusivo de X2. La estación AN, con 10.100 nulos, está totalmente vacía.

En 2024 y 2025 los nulos de temperatura y humedad escalan hasta igualarse con los de irradiancia y viento (aproximadamente 20.249), lo que confirma la inactividad progresiva de X2, que en 2025 deja de reportar prácticamente todas las variables.

### <font color='#C0392B'><b>1.3 Cobertura de estaciones de meteocat Barcelona</b></font>


In [ ]:
df.group_by(["codi_estacio", "anio"]).agg([
    pl.len().alias("total_registros"),
    pl.col("temp_mean").is_null().sum().alias("nulos_temp"),
    pl.col("viento_mean").is_null().sum().alias("nulos_viento"),
    pl.col("irradiancia_mean").is_null().sum().alias("nulos_irradiancia"),
]).sort(["codi_estacio", "anio"])

In [ ]:
(
    df.group_by("codi_estacio")
    .agg([
        pl.len().alias("total"),
        (pl.col("temp_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_temp"),
        (pl.col("humedad_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_humedad"),
        (pl.col("viento_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_viento"),
        (pl.col("irradiancia_mean").is_null().sum() / pl.len() * 100).round(2).alias("pct_irradiancia"),
        (pl.col("lst_celsius").is_null().sum() / pl.len() * 100).round(2).alias("pct_lst"),
    ])
    .sort("codi_estacio")
)

La cobertura de nulos por estación es la siguiente:

- X4: 0.57% nulos
- X8: 0.62% nulos
- D5: 0.80% nulos
- X2: 17.23% nulos
- AN: 100% nulos

Estación AN, Parc de la Ciutadella: AN y X2 se solapan físicamente, a menos de 200 metros dentro del mismo parque. Más importante aún, AN tiene el 100% de nulos en todas sus variables meteorológicas en todos los años, de 2019 a 2025, así que existe como punto de referencia geográfico en Meteocat pero no dispone de sensores activos. El paso a seguir será reasignar todos los registros originalmente asignados a AN hacia X2, que sí tiene datos de temperatura.

Estación X2, Zoo de Barcelona: X2 presenta dos limitaciones. En primer lugar, nunca ha tenido sensores de viento ni irradiancia, ya que ambas variables tienen el 100% de nulos en todos los años. En segundo lugar, a partir de 2024 comienza a fallar y en 2025 deja de funcionar completamente, con el 100% de nulos en temperatura. Para los códigos postales asignados a X2, viento e irradiancia se imputarán desde X4 en feature engineering, y la temperatura desde X4 con un factor de corrección histórico (ratio X2/X4, 2019–2023).

# <font color='#1B3A5C'>  **2. Limpieza inicial** </font>

A partir de los hallazgos anteriores se comienza con una limpieza inicial.

### <font color='#C0392B'><b>2.1 Trato de Datos Faltantes</b></font>

#### <font color='#2A9D8F'><b>2.1.1 Datos sin registros en mwh_total</b></font>

In [ ]:
mwh_problematicos = df.filter(pl.col('mwh_total') <= 0)
print(f"Registros con mwh_total mayor o igual a 0: {len(mwh_problematicos):,}")

n_dupes = len(df) - df.unique(subset=['datetime', 'cod_postal']).shape[0]
print(f"Duplicados datetime + cod_postal: {n_dupes:,}")

Existen 378 filas donde no hay valores iguales o menores a 0 en mwh_total, lo cual puede tratarse de un problema de recolección de datos.

In [ ]:
print(mwh_problematicos.select([
    "cod_postal", "nombre_postal", "datetime",
    "mwh_total", "mwh_industria", "mwh_residencial",
    "mwh_servicios", "mwh_no_especificado"
]))

In [ ]:
print(mwh_problematicos.select("datetime").unique().sort("datetime"))

In [ ]:
fechas_problema = ["2025-06-26", "2025-08-30", "2025-09-07"]

df.filter(
    (pl.col("datetime").dt.date().cast(pl.Utf8).is_in(fechas_problema)) &
    (pl.col("datetime").dt.hour() == 18)
).select(["cod_postal", "datetime", "mwh_total"]).head(10)

In [ ]:
mwh_problematicos.group_by("datetime") \
    .agg(pl.n_unique("cod_postal").alias("n_codigos")) \
    .sort("datetime")

Este error en los datos existe durante todos los días para los 42 códigos postales.

Se detectaron 378 registros con mwh_total = 0, concentrados en exactamente 3 fechas de 2025 (26-jun, 30-ago, 07-sep). En cada caso, los bloques 00:00, 06:00 y 12:00 de los 42 códigos postales reportaron cero simultáneamente, mientras que el bloque 18:00 del mismo día registró valores normales. Es un fallo parcial de OpenDataBCN, no un corte real de suministro.

#### <font color='#2A9D8F'><b>2.1.2 Verificar continuidad temporal</b></font>

In [ ]:
fechas_esperadas = df.select('cod_postal').unique().join(
    pl.datetime_range(df['datetime'].min(), df['datetime'].max(), interval='6h', eager=True)
      .alias('datetime').to_frame(), how='cross'
)
faltantes = fechas_esperadas.join(df.select(['cod_postal', 'datetime']), on=['cod_postal', 'datetime'], how='anti')
print(f"Faltantes detectados: {len(faltantes):,}")
print(faltantes.filter(pl.col('datetime').dt.date() != date(2025, 8, 19))
               .group_by('cod_postal').agg(pl.len().alias('n')).sort('n', descending=True))

docs_festivos = list(db["clean_festivos"].find({}, {"_id": 0}))
fechas_festivas = set(
    pl.DataFrame(docs_festivos)
    .with_columns(pl.col("fecha").str.to_date("%Y-%m-%d"))["fecha"].to_list()
)

referencia = (
    df.filter(pl.col("anio").is_in([2022, 2023, 2024]))
    .group_by(["cod_postal", "hora", "dia_semana", "mes"])
    .agg(pl.col("mwh_total").median().alias("mwh_total"))
)

estaticas = (
    df.filter(pl.col("nombre_postal").is_not_null())
    .select(["cod_postal", "nombre_postal", "centroide_lon", "centroide_lat",
             "codi_estacio", "nombre_estacio", "estacio_lon", "estacio_lat"])
    .unique(subset=["cod_postal"])
)

df_faltantes = (
    faltantes
    .with_columns([
        pl.col("datetime").dt.hour().cast(pl.Int8).alias("hora"),
        pl.col("datetime").dt.weekday().cast(pl.Int8).alias("dia_semana"),
        pl.col("datetime").dt.month().cast(pl.Int8).alias("mes"),
        pl.col("datetime").dt.year().cast(pl.Int16).alias("anio"),
        pl.col("datetime").dt.week().cast(pl.Int8).alias("semana_anio"),
        (pl.col("datetime").dt.weekday() >= 6).cast(pl.Int8).alias("es_finde"),
    ])
    .join(referencia, on=["cod_postal", "hora", "dia_semana", "mes"], how="left")
    .join(estaticas, on="cod_postal", how="left")
    .with_columns(
        pl.col("datetime").dt.date()
          .map_elements(lambda d: 1 if d in fechas_festivas else 0, return_dtype=pl.Int64)
          .alias("es_festivo")
    )
)

df = pl.concat([df, df_faltantes], how="diagonal_relaxed").sort(["cod_postal", "hora", "datetime"])
print(f"Shape: {df.shape} | Nulls mwh_total: {df['mwh_total'].null_count()}")
print(f"Nulls nombre_postal: {df['nombre_postal'].null_count()}")

In [ ]:
faltantes_por_dia = faltantes.with_columns(
    pl.col("datetime").dt.date().alias("fecha")
).group_by(["cod_postal", "fecha"]).count()

faltantes_por_dia.filter(
    pl.col("count") != 4
)

A todos estos días les faltan los 4 horarios. El código postal 08011 (7–20 ago 2025) tiene 52 bloques faltantes (13 días por 4 bloques), un problema de reporte específico de ese CP. El 2025-08-19 es un día completo sin datos para los 42 códigos postales (168 bloques), un fallo de reporte en la fuente.

### <font color='#C0392B'><b>2.2 Conversión de tipo de datos</b></font>

In [ ]:
df = df.with_columns([
    pl.col('datetime').cast(pl.Datetime),
    pl.col('cod_postal').cast(pl.Utf8),
    pl.col('es_festivo').cast(pl.Int8),
    pl.col('es_finde').cast(pl.Int8),
    pl.col('hora').cast(pl.Int8),
    pl.col('dia_semana').cast(pl.Int8),
    pl.col('mes').cast(pl.Int8),
    pl.col('anio').cast(pl.Int16),
])
df.head(3)

### <font color='#C0392B'><b>2.3 Trato de Nulls</b></font>

El tratamiento de nulos sigue cinco pasos. Primero, la reasignación de AN a X2, con las variables meteorológicas rellenadas con datos reales de X2 desde MongoDB. Segundo, la construcción de una grilla temporal completa, donde los bloques faltantes (08011 en agosto de 2025 y el 19-ago global) se rellenan con la mediana histórica. Tercero, los sectores de consumo mwh_industria, mwh_residencial y mwh_servicios se imputan a 0 (sector no activo en ese bloque) y se recalcula mwh_total. Cuarto, los días con mwh_total = 0, correspondientes a 3 fechas de fallo de reporte, se imputan con la mediana histórica 2019-2024. Por último, los nulos meteorológicos restantes (viento, irradiancia, temperatura y humedad) son estructurales y se mantienen para feature engineering.

#### <font color='#2A9D8F'><b>2.3.1 Estrategia de nulos meteorológicos</b></font>

Para viento_mean, irradiancia_mean y precipitacion_sum, X2 no mide estas variables en ningún período. Se mantienen los nulos; si el EDA muestra diferencias significativas entre barrios, se tratarán en feature engineering.

Para temp_mean y humedad_mean en el período 2024 parcial y 2025, X2 está inactiva y solo hay datos reales disponibles para 2019–2023. Se mantienen los nulos por ahora; si el EDA confirma diferencias de comportamiento entre barrios (consumo frente a temperatura), se aplicará un factor de corrección basado en la relación histórica X2/X4 de años anteriores.

El criterio de decisión posterior al EDA es el siguiente: si la variación entre barrios es baja, se dejan los nulos sin imputar y el modelo los gestiona; si la variación entre barrios es alta, se aplica el factor de corrección histórico X2/X4.

#### <font color='#2A9D8F'><b>2.3.2 Reasignación AN a X2</b></font>

In [ ]:
df = df.with_columns([
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit("X2")).otherwise(pl.col("codi_estacio")).alias("codi_estacio"),
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit("Barcelona - Zoo")).otherwise(pl.col("nombre_estacio")).alias("nombre_estacio"),
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit(2.1898)).otherwise(pl.col("estacio_lon")).alias("estacio_lon"),
    pl.when(pl.col("codi_estacio") == "AN").then(pl.lit(41.3842)).otherwise(pl.col("estacio_lat")).alias("estacio_lat"),
])

docs_meteocat = list(db["clean_meteocat"].find({"codi_estacio": "X2"}, {"_id": 0}))
meteocat_x2 = (
    pl.DataFrame(docs_meteocat)
    .rename({"data_lectura": "datetime", "temp": "temp_mean", "humedad": "humedad_mean",
             "viento": "viento_mean", "precipitacion": "precipitacion_sum", "irradiancia": "irradiancia_mean"})
    .select(["datetime", "temp_mean", "humedad_mean", "viento_mean", "precipitacion_sum", "irradiancia_mean"])
)

df = df.join(meteocat_x2, on="datetime", how="left", suffix="_x2")

cols_meteo = ["temp_mean", "humedad_mean", "viento_mean", "precipitacion_sum", "irradiancia_mean"]
df = df.with_columns([
    pl.when(pl.col(c).is_null()).then(pl.col(f"{c}_x2")).otherwise(pl.col(c)).alias(c)
    for c in cols_meteo
]).drop([f"{c}_x2" for c in cols_meteo])

print("Reasignación AN a X2 completada")
print(df.group_by("codi_estacio").agg(pl.col("temp_mean").null_count().alias("nulos_temp"), pl.len().alias("total")).sort("codi_estacio"))

#### <font color='#2A9D8F'><b>2.3.3 Grilla temporal completa</b></font>

El código postal 08011 tiene 52 bloques faltantes en agosto de 2025 y el 19-ago-2025 falta para todos los códigos postales. Se imputan con la mediana histórica por CP, hora, día de la semana y mes (2022-2024).

In [ ]:
fechas_esperadas = df.select('cod_postal').unique().join(
    pl.datetime_range(df['datetime'].min(), df['datetime'].max(), interval='6h', eager=True)
      .alias('datetime').to_frame(), how='cross'
)
faltantes = fechas_esperadas.join(df.select(['cod_postal', 'datetime']), on=['cod_postal', 'datetime'], how='anti')
print(f"Faltantes detectados: {len(faltantes):,}")
print(faltantes.filter(pl.col('datetime').dt.date() != date(2025, 8, 19))
               .group_by('cod_postal').agg(pl.len().alias('n')).sort('n', descending=True))

docs_festivos = list(db["clean_festivos"].find({}, {"_id": 0}))
fechas_festivas = set(
    pl.DataFrame(docs_festivos)
    .with_columns(pl.col("fecha").str.to_date("%Y-%m-%d"))["fecha"].to_list()
)

referencia = (
    df.filter(pl.col("anio").is_in([2022, 2023, 2024]))
    .group_by(["cod_postal", "hora", "dia_semana", "mes"])
    .agg(pl.col("mwh_total").median().alias("mwh_total"))
)

df_faltantes = (
    faltantes
    .with_columns([
        pl.col("datetime").dt.hour().cast(pl.Int8).alias("hora"),
        pl.col("datetime").dt.weekday().cast(pl.Int8).alias("dia_semana"),
        pl.col("datetime").dt.month().cast(pl.Int8).alias("mes"),
        pl.col("datetime").dt.year().cast(pl.Int16).alias("anio"),
        pl.col("datetime").dt.week().cast(pl.Int8).alias("semana_anio"),
        (pl.col("datetime").dt.weekday() >= 6).cast(pl.Int8).alias("es_finde"),
    ])
    .join(referencia, on=["cod_postal", "hora", "dia_semana", "mes"], how="left")
    .with_columns(
        pl.col("datetime").dt.date()
          .map_elements(lambda d: 1 if d in fechas_festivas else 0, return_dtype=pl.Int64)
          .alias("es_festivo")
    )
)

df = pl.concat([df, df_faltantes], how="diagonal_relaxed").sort(["cod_postal", "hora", "datetime"])
print(f"Shape: {df.shape} | Nulls mwh_total: {df['mwh_total'].null_count()}")

El código postal 08011 (7–20 ago 2025) presenta 52 bloques faltantes (13 días por 4 bloques), un problema de reporte específico del CP. El 2025-08-19 es un día completo sin datos para los 42 códigos postales (168 bloques), un fallo de OpenDataBCN. Ambos se imputan con la mediana histórica por CP, hora, día de la semana y mes, usando 2022-2024.

#### <font color='#2A9D8F'><b>2.3.4 Fill null sectores + recálculo mwh_total</b></font>

Los sectores sin dato se ponen a 0 (sector no activo en ese bloque horario).

In [ ]:
df = df.with_columns([
    pl.col("mwh_industria").fill_null(0),
    pl.col("mwh_residencial").fill_null(0),
    pl.col("mwh_servicios").fill_null(0),
])

df = df.with_columns(
    pl.when(pl.col("mwh_total").is_null())
    .then(
        pl.col("mwh_industria") + pl.col("mwh_residencial") +
        pl.col("mwh_servicios") + pl.col("mwh_no_especificado").fill_null(0)
    )
    .otherwise(pl.col("mwh_total"))
    .alias("mwh_total")
)
nulos = (
    df.null_count()
    .unpivot(variable_name="columna", value_name="nulos")
    .filter(pl.col("nulos") > 0)
    .with_columns((pl.col("nulos") / len(df) * 100).round(2).alias("pct"))
    .sort("nulos", descending=True)
)
print(f"Shape: {df.shape}")
print(nulos)

#### <font color='#2A9D8F'><b>2.3.5 Imputar mwh_total = 0</b></font>

El 26-jun, 30-ago y 07-sep de 2025, mwh_total = 0 para los 42 códigos postales por un error en OpenData. Se imputa con la mediana histórica por CP, hora, día de la semana y mes (2022-2024).

In [ ]:
referencia_ceros = (
    df.filter(pl.col("anio").is_in([2022, 2023, 2024]) & (pl.col("mwh_total") > 0))
    .group_by(["cod_postal", "hora", "dia_semana", "mes"])
    .agg(pl.col("mwh_total").median().alias("mwh_ref"))
)

df = (
    df.join(referencia_ceros, on=["cod_postal", "hora", "dia_semana", "mes"], how="left")
    .with_columns(
        pl.when(pl.col("mwh_total") == 0)
          .then(pl.col("mwh_ref"))
          .otherwise(pl.col("mwh_total"))
          .alias("mwh_total")
    )
    .drop("mwh_ref")
)

print(f"Ceros restantes: {df.filter(pl.col('mwh_total') == 0).shape[0]}")
print(f"Nulls mwh_total: {df['mwh_total'].null_count()}")

El tratamiento de nulos aplicado en el EDA se resume a continuación. Los registros de la estación AN se reasignaron a X2 y se rellenaron con datos reales de Meteocat para temperatura y humedad (2019–2023). Los nulos de mwh_industria, mwh_residencial y mwh_servicios se imputaron a 0 (sector no activo en ese bloque), y mwh_total se recalculó como la suma de los 4 sectores. En cuanto a la continuidad temporal, se añadieron 220 bloques: 52 del CP 08011 (7–20 ago 2025, fallo de reporte específico) y 168 del 19-ago-2025 global (42 códigos por 4 bloques), imputados con la mediana histórica por CP, hora, día de la semana y mes (2022–2024). Para mwh_total = 0 se trataron 378 registros en 3 fechas de 2025 (26-jun, 30-ago, 07-sep): un fallo en OpenData con los bloques 00:00, 06:00 y 12:00 en cero mientras el de 18:00 era normal, imputados con la mediana histórica por CP, hora, día de la semana y mes (2022–2024). Los nulos meteorológicos restantes, de viento e irradiancia (X2 sin sensores en todo el período) y de temperatura y humedad en 2024–2025 (X2 inactiva), son conocidos y se mantienen para feature engineering.

Como limitación conocida, los 378 registros con mwh_total = 0 imputados con mediana histórica presentan inconsistencia con sus sectores componentes (mwh_industria, mwh_residencial y mwh_servicios permanecen en 0). Al representar el 0.09% del dataset y no usarse los sectores como features del modelo, el impacto en el forecasting es despreciable. Se documenta como deuda técnica para versiones futuras.

---
# <font color='#1B3A5C'>  **3. Análisis Descriptivo de la Variable Objetivo: mwh_total** </font>

In [ ]:
df.select('mwh_total').describe()

La media (101,467 MWh) es mayor que la mediana (90,608 MWh), con una diferencia de aproximadamente 11k, lo que confirma un sesgo positivo: la cola derecha de outliers extremos arrastra la media hacia arriba. El IQR va de 57,120 a 132,708 MWh, el rango de consumo normal entre barrios y bloques horarios. El mínimo de 130 MWh corresponde a registros de madrugada en barrios pequeños, un valor plausible, mientras que el máximo de 1,486,114 MWh es un outlier extremo a revisar. La desviación estándar (59,563) representa una dispersión alta, esperable dado que mezcla 42 códigos postales con perfiles muy distintos (industrial, residencial, turístico).

In [ ]:
vals_target = df['mwh_total'].drop_nulls().to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(vals_target, bins=80, color='#2A9D8F', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución de mwh_total', fontweight='bold')
axes[0].set_xlabel('MWh')
axes[0].set_ylabel('Frecuencia')

axes[1].boxplot(vals_target, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#2A9D8F', alpha=0.7),
                medianprops=dict(color='navy', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#E9C46A'))
axes[1].set_title('Boxplot de mwh_total', fontweight='bold')
axes[1].set_xlabel('MWh')

plt.tight_layout()
plt.show()

In [ ]:
print(f"sum(is.na(mwh_total)): {df['mwh_total'].null_count()}")

La distribución de mwh_total presenta sesgo positivo con una cola derecha pronunciada. La mediana está muy por debajo de la media, lo que indica que la mayoría de los bloques tienen un consumo moderado pero existen picos elevados de alta magnitud. Los outliers identificados son de 1.48 M, lo que muestra que no pueden ser datos reales. La desviación estándar de 59 k refleja la alta dispersión: los valores están muy dispersos.

No se sigue una distribución normal, así que se usará Spearman para las correlaciones y Kruskal-Wallis y Mann-Whitney para los tests de hipótesis.

### <font color='#C0392B'><b>3.1 Outliers</b></font>

La detección emplea un umbral de 5 veces la mediana por CP. Se usa la mediana porque no se infla por los propios outliers que intenta detectar.

In [ ]:
umbrales = df.group_by("cod_postal").agg(pl.col("mwh_total").median().alias("mediana"))

print(df.join(umbrales, on="cod_postal")
    .filter(pl.col("mwh_total") > pl.col("mediana") * 5)
    .group_by("cod_postal")
    .agg(pl.len().alias("n_registros"))
    .sort("n_registros", descending=True))

Seis códigos superan el umbral de 5 veces su mediana. El 08037 destaca con 450 registros, muy por encima del resto. Se revisa caso a caso.

In [ ]:

casos = {
    "08030 — Sostenido (may 2023)": df.filter(
        (pl.col("cod_postal") == "08030") &
        (pl.col("datetime") >= pl.datetime(2023, 5, 5)) &
        (pl.col("datetime") < pl.datetime(2023, 5, 12))
    ).select(["datetime", "mwh_total", "mwh_industria"]).sort("datetime"),
    "08037 — Puntual (nov/dic 2024, feb 2025)": df.filter(
        (pl.col("cod_postal") == "08037") &
        ((pl.col("datetime") == pl.datetime(2024, 11, 20, 0)) |
         (pl.col("datetime") == pl.datetime(2024, 12, 22, 0)) |
         (pl.col("datetime") == pl.datetime(2025, 2, 24, 0)))
    ).select(["datetime", "mwh_total", "mwh_servicios"]),
    "08022 — Parcial (may 2023)": df.filter(
        (pl.col("cod_postal") == "08022") &
        (pl.col("datetime") >= pl.datetime(2023, 5, 29)) &
        (pl.col("datetime") < pl.datetime(2023, 5, 31))
    ).select(["datetime", "mwh_total", "mwh_residencial"]).sort("datetime"),
}
for nombre, resultado in casos.items():
    print(f"\n{'='*60}\n  {nombre}\n{'='*60}")
    print(resultado)

In [ ]:
print("\n" + "="*60 + "\n  Valores > 400k\n" + "="*60)
print(df.filter(pl.col("mwh_total") > 400000)
    .select(["datetime", "cod_postal", "nombre_postal", "mwh_total"])
    .sort("mwh_total", descending=True).head(10))

for cp in ["08013", "08036", "08009", "08006", "08011", "08030"]:
    fecha_max = df.filter(pl.col("cod_postal") == cp).sort("mwh_total", descending=True).head(1)["datetime"][0]
    print(f"\n{'='*50}\nCP {cp}\n{'='*50}")
    print(df.filter(
        (pl.col("cod_postal") == cp) &
        (pl.col("datetime") >= fecha_max - pl.duration(days=3)) &
        (pl.col("datetime") <= fecha_max + pl.duration(days=3))
    ).select(["datetime", "mwh_total"]).sort("datetime"))

Errores de reporte, se imputan con la mediana de los 3 años limpios anteriores a cada caso (excluyendo el año COVID 2020 y sin usar datos futuros, imputación causal):

- 08030 (5–11 may 2023, 28 bloques): mwh_industria de 6 a 12 veces el habitual, aproximadamente 50,000 durante 7 días consecutivos. La caída brusca al día siguiente sin gradualidad es un fallo de acumulación en el reporte.
- 08037 (nov-2024, dic-2024, feb-2025, 3 bloques): mwh_servicios de 1.46M frente a aproximadamente 35k habitual. Patrón idéntico en los 3 casos en el mismo bloque horario (00:00h), posible error de acumulación.
- 08022 (29–30 may 2023, 6 bloques): bloques 00:00h normales pero 06:00, 12:00 y 18:00h inflados de 10 a 15 veces. Posible error de distribución del consumo diario entre bloques.
- 08030 (9 ene 2025, 4 bloques): salto abrupto de unos 300k a 558-595k en un solo día sin contexto climático que lo justifique. Vuelta a la normalidad al día siguiente.
- 08037 (jul–nov 2025, 475 bloques): todos los sectores de 6 a 7 veces el histórico, entre 80-92k MWh durante 5 meses consecutivos. Imposible que sea demanda real.
- 08009 (2 jun 2025, 4 bloques): todos los sectores inflados proporcionalmente (aproximadamente 7 veces lo habitual) en un único día, sin contexto.

Picos reales, se mantienen:

- 08013 (jul 2025): crecimiento gradual durante varios días (aproximadamente 563k máx). Patrón consistente con una ola de calor de inicio de verano.
- 08036 (ago 2023): valores sostenidos 3-4 días (aproximadamente 535k máx). Gradualidad clara de subida y bajada, consistente con una ola de calor.
- 08006 (jul 2025): pico de 4 a 5 días con valores crecientes y decrecientes. Mismo patrón de ola de calor que 08013.
- 08011 (ene 2020): valores altos sostenidos durante más de una semana.

### <font color='#C0392B'><b>3.2 Imputación de Outliers</b></font>

Funciones de imputación por la mediana de los 3 años limpios anteriores a cada outlier (excluyendo 2020 por COVID y sin usar datos futuros, evitando así el data leakage).

In [ ]:
def anios_ref_3(anio):
    refs, y = [], anio - 1
    while len(refs) < 3 and y >= 2019:
        if y != 2020:
            refs.append(y)
        y -= 1
    return sorted(refs)

def imputar_sector(df, cp, sector, filtro, anios_ref, keys=["hora", "mes"]):
    med = (
        df.filter((pl.col("cod_postal") == cp) & ~filtro & pl.col("anio").is_in(anios_ref))
        .group_by(keys)
        .agg(pl.col(sector).median().alias("_med"))
    )
    df = (df.join(med, on=keys, how="left")
        .with_columns(
            pl.when((pl.col("cod_postal") == cp) & filtro)
              .then(pl.col("_med")).otherwise(pl.col(sector)).alias(sector)
        ).drop("_med"))
    return df.with_columns(
        pl.when((pl.col("cod_postal") == cp) & filtro)
        .then(
            pl.col("mwh_industria") + pl.col("mwh_residencial") +
            pl.col("mwh_servicios") + pl.col("mwh_no_especificado").fill_null(0)
        )
        .otherwise(pl.col("mwh_total"))
        .alias("mwh_total")
    )

def imputar_todos_sectores(df, cp, cond, anios_ref, mes_ref, keys=["hora"]):
    mes_filter = pl.col("mes").is_in(mes_ref) if isinstance(mes_ref, list) else pl.col("mes") == mes_ref
    med = (
        df.filter((pl.col("cod_postal") == cp) & pl.col("anio").is_in(anios_ref) & mes_filter)
        .group_by(keys)
        .agg([pl.col("mwh_industria").median().alias("_ind"),
              pl.col("mwh_residencial").median().alias("_res"),
              pl.col("mwh_servicios").median().alias("_ser")])
    )
    df = (df.join(med, on=keys, how="left")
        .with_columns([
            pl.when(cond).then(pl.col("_ind")).otherwise(pl.col("mwh_industria")).alias("mwh_industria"),
            pl.when(cond).then(pl.col("_res")).otherwise(pl.col("mwh_residencial")).alias("mwh_residencial"),
            pl.when(cond).then(pl.col("_ser")).otherwise(pl.col("mwh_servicios")).alias("mwh_servicios"),
        ]).drop(["_ind", "_res", "_ser"]))
    return df.with_columns(
        pl.when(cond)
        .then(
            pl.col("mwh_industria") + pl.col("mwh_residencial") +
            pl.col("mwh_servicios") + pl.col("mwh_no_especificado").fill_null(0)
        )
        .otherwise(pl.col("mwh_total"))
        .alias("mwh_total")
    )

df = imputar_sector(df, "08030", "mwh_industria",
    (pl.col("datetime") >= pl.datetime(2023, 5, 5)) & (pl.col("datetime") < pl.datetime(2023, 5, 12)),
    anios_ref=anios_ref_3(2023))

df = imputar_sector(df, "08037", "mwh_servicios",
    (pl.col("datetime") == pl.datetime(2024, 11, 20, 0)) |
    (pl.col("datetime") == pl.datetime(2024, 12, 22, 0)),
    anios_ref=anios_ref_3(2024))
df = imputar_sector(df, "08037", "mwh_servicios",
    (pl.col("datetime") == pl.datetime(2025, 2, 24, 0)),
    anios_ref=anios_ref_3(2025))

df = imputar_sector(df, "08022", "mwh_residencial",
    (pl.col("datetime") >= pl.datetime(2023, 5, 29)) & (pl.col("datetime") < pl.datetime(2023, 5, 31)),
    anios_ref=anios_ref_3(2023))

df = imputar_todos_sectores(df, "08030",
    cond=(pl.col("cod_postal") == "08030") & (pl.col("datetime").dt.date() == pl.date(2025, 1, 9)),
    anios_ref=anios_ref_3(2025), mes_ref=1)

df = imputar_todos_sectores(df, "08037",
    cond=(pl.col("cod_postal") == "08037") & (pl.col("anio") == 2025) &
         (pl.col("mes").is_in([7, 8, 9, 10, 11])) & (pl.col("mwh_total") > 100000),
    anios_ref=anios_ref_3(2025), mes_ref=[7, 8, 9, 10, 11], keys=["hora", "mes", "dia_semana"])

_cond_noesp = (
    (pl.col("cod_postal") == "08037") & (pl.col("anio") == 2025) &
    (pl.col("mes").is_in([7, 8, 9, 10, 11])) & (pl.col("mwh_no_especificado").is_not_null())
)
df = df.with_columns(
    pl.when(_cond_noesp)
    .then(pl.lit(None).cast(pl.Int64))
    .otherwise(pl.col("mwh_no_especificado"))
    .alias("mwh_no_especificado")
)
df = df.with_columns(
    pl.when(_cond_noesp)
    .then(pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios"))
    .otherwise(pl.col("mwh_total"))
    .alias("mwh_total")
)

df = imputar_todos_sectores(df, "08009",
    cond=(pl.col("cod_postal") == "08009") & (pl.col("datetime").dt.date() == pl.date(2025, 6, 2)),
    anios_ref=anios_ref_3(2025), mes_ref=6)

print("Todos los outliers imputados")

In [ ]:
df = df.with_columns(
    pl.when(pl.col("mwh_total").is_null())
    .then(
        pl.col("mwh_industria").fill_null(0) + pl.col("mwh_residencial").fill_null(0) +
        pl.col("mwh_servicios").fill_null(0) + pl.col("mwh_no_especificado").fill_null(0)
    )
    .otherwise(pl.col("mwh_total"))
    .alias("mwh_total")
)

print(f"Valores > 400k restantes: {df.filter(pl.col('mwh_total') > 400000).shape[0]}")
print(df.filter(pl.col("mwh_total") > 400000)
    .select(["datetime", "cod_postal", "nombre_postal", "mwh_total", "mwh_industria", "mwh_residencial", "mwh_servicios"])
    .sort("mwh_total", descending=True).head(20))

vals_target = df['mwh_total'].drop_nulls().to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(vals_target, bins=80, color='#1B3A5C', edgecolor='white', linewidth=0.3, alpha=0.85)
axes[0].set_title('Distribución de mwh_total (tras imputación)', fontweight='bold')
axes[0].set_xlabel('MWh'); axes[0].set_ylabel('Frecuencia')
axes[1].boxplot(vals_target, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#1B3A5C', alpha=0.7),
                medianprops=dict(color='#E76F51', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#1B3A5C'),
                whiskerprops=dict(color='#1B3A5C'), capprops=dict(color='#1B3A5C'))
axes[1].set_title('Boxplot de mwh_total (tras tratamiento outliers)', fontweight='bold')
axes[1].set_xlabel('MWh')
plt.tight_layout(); plt.show()

Tras imputar los 520 registros erróneos con la mediana histórica (2022-2024), la distribución muestra una forma log-normal coherente con el comportamiento esperado del consumo eléctrico urbano. La mayoría de los bloques se concentran entre 20k y 150k MWh, con una cola derecha moderada que refleja los picos reales de demanda estival. Los valores máximos legítimos (olas de calor en 08013, 08036 y 08006; ola de frío en 08011) se mantienen como parte de la variabilidad natural del sistema.

### <font color='#C0392B'><b>3.3 Descomposición de consumo energético</b></font>


In [ ]:
serie_global = (
    df.group_by("datetime")
    .agg([
        pl.col("mwh_industria").sum(),
        pl.col("mwh_residencial").sum(),
        pl.col("mwh_servicios").sum(),
    ])
    .sort("datetime")
)

fechas = serie_global["datetime"].to_numpy()
ind    = serie_global["mwh_industria"].to_numpy()
res    = serie_global["mwh_residencial"].to_numpy()
ser    = serie_global["mwh_servicios"].to_numpy()

fig, ax = plt.subplots(figsize=(24, 5))
ax.stackplot(fechas, ind, res, ser,
             labels=["Industria", "Residencial", "Servicios"],
             colors=["#264653", "#2A9D8F", "#E9C46A"],
             alpha=0.8)
ax.set_title("Consumo por sector — Barcelona (todos los CPs)", fontweight="bold")
ax.set_ylabel("MWh")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
(
    df.group_by("cod_postal")
    .agg([
        pl.col("mwh_industria").sum(),
        pl.col("mwh_residencial").sum(),
        pl.col("mwh_servicios").sum(),
        pl.col("mwh_no_especificado").sum(),
    ])
    .with_columns([
        (pl.col("mwh_industria") / (pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios")) * 100).round(1).alias("pct_industria"),
        (pl.col("mwh_residencial") / (pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios")) * 100).round(1).alias("pct_residencial"),
        (pl.col("mwh_servicios") / (pl.col("mwh_industria") + pl.col("mwh_residencial") + pl.col("mwh_servicios")) * 100).round(1).alias("pct_servicios"),
    ])
    .select(["cod_postal", "pct_industria", "pct_residencial", "pct_servicios"])
    .sort("pct_industria", descending=True)
)

mwh_total es una variable compuesta: es la suma del consumo industrial, residencial, de servicios y no especificado. Ningún código postal tiene un perfil puro. Servicios domina en 38 de 42 CPs, siendo el sector principal en toda Barcelona. La industria solo es relevante en 3 o 4 CPs (08039, 08004 y 08038 principalmente). El residencial alcanza su máximo en zonas periféricas como 08032 (70% residencial). Esta composición explica por qué los patrones temporales difieren entre barrios: según el código postal será más probable tener picos en distintos horarios.

### <font color='#C0392B'><b>3.4 Análisis de Serie Temporal de Consumo</b></font>


In [ ]:

serie_global = (
    df.group_by("datetime")
    .agg([
        pl.col("mwh_industria").sum(),
        pl.col("mwh_residencial").sum(),
        pl.col("mwh_servicios").sum(),
    ])
    .sort("datetime")
)

fechas = serie_global["datetime"].to_numpy()
ind    = serie_global["mwh_industria"].to_numpy()
res    = serie_global["mwh_residencial"].to_numpy()
ser    = serie_global["mwh_servicios"].to_numpy()

fig, ax = plt.subplots(figsize=(24, 5))
ax.stackplot(fechas, ind, res, ser,
             labels=["Industria", "Residencial", "Servicios"],
             colors=["#264653", "#2A9D8F", "#E9C46A"],
             alpha=0.8)
ax.set_title("Consumo por sector — Barcelona (todos los CPs)", fontweight="bold")
ax.set_ylabel("MWh")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

El consumo eléctrico de Barcelona muestra una estacionalidad anual clara, con picos en verano (julio–agosto) y en invierno (diciembre–enero), consistente con el efecto combinado del aire acondicionado y la calefacción. Los servicios dominan el consumo en todos los períodos, seguidos del residencial, y la industria tiene un peso estable.

Se observa una caída notable en 2020 (COVID-19) y una recuperación progresiva hasta 2022. A partir de 2023 el consumo se estabiliza con niveles ligeramente superiores al período prepandemia.

In [ ]:
(df.group_by("cod_postal")
 .agg([
     pl.col("nombre_postal").drop_nulls().first().alias("nombre_postal"),
     pl.len().alias("n_registros"),
     pl.col("mwh_total").null_count().alias("nulls"),
     pl.col("mwh_total").mean().alias("mwh_mean"),
     pl.col("mwh_industria").sum().alias("sum_industria"),
     pl.col("mwh_residencial").sum().alias("sum_residencial"),
     pl.col("mwh_servicios").sum().alias("sum_servicios"),
     pl.col("mwh_total").sum().alias("sum_total"),
 ])
 .with_columns([
     ((pl.col("sum_industria") / pl.col("sum_total") * 100).round(1).cast(pl.Utf8) + pl.lit("%")).alias("pct_industria"),
     ((pl.col("sum_residencial") / pl.col("sum_total") * 100).round(1).cast(pl.Utf8) + pl.lit("%")).alias("pct_residencial"),
     ((pl.col("sum_servicios") / pl.col("sum_total") * 100).round(1).cast(pl.Utf8) + pl.lit("%")).alias("pct_servicios"),
 ])
 .drop(["sum_industria", "sum_residencial", "sum_servicios", "sum_total"])
 .sort("mwh_mean", descending=True)
)

In [ ]:
medianas = df.group_by("cod_postal").agg(pl.col("mwh_total").median().alias("mediana_cp"))

df.filter(pl.col("mwh_total") > 400000)\
  .group_by(["cod_postal", "nombre_postal"])\
  .agg([
      pl.len().alias("n_registros"),
      pl.col("mwh_total").max().alias("max_mwh"),
  ])\
  .join(medianas, on="cod_postal")\
  .sort("n_registros", descending=True)

Se seleccionan 4 códigos postales representativos de perfiles de consumo distintos, con cobertura completa (10.104 registros) y mínimos nulos:

- 08038 Montjuïc / Zona Franca, por su perfil industrial (36.7% industria), la mayor proporción del dataset.
- 08005 El Poblenou, por su perfil mixto equilibrado, referente en la literatura de smart city de Barcelona, con 4 nulos.
- 08032 El Carmel / El Guinardó, por su perfil residencial dominante (70.3% residencial).
- 08002 Barri Gòtic, por su perfil de servicios/turístico (75.6% servicios) y consumo estacional marcado.

In [ ]:
CPS = {
    "08038": "Montjuïc / Zona Franca (Industrial)",
    "08005": "El Poblenou (Mixto)",
    "08032": "El Carmel / El Guinardó (Residencial)",
    "08002": "Barri Gòtic (Servicios/Turístico)",
}

In [ ]:
for cp, nombre in CPS.items():
    serie_cp = (
        df.filter(pl.col("cod_postal") == cp)
        .sort("datetime")
        .select(["datetime", "mwh_industria", "mwh_residencial", "mwh_servicios"])
    )

    fechas = serie_cp["datetime"].to_numpy()
    ind    = serie_cp["mwh_industria"].to_numpy()
    res    = serie_cp["mwh_residencial"].to_numpy()
    ser    = serie_cp["mwh_servicios"].to_numpy()

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.stackplot(fechas, ind, res, ser,
                 labels=["Industria", "Residencial", "Servicios"],
                 colors=["#264653", "#2A9D8F", "#E9C46A"],
                 alpha=0.8)
    ax.set_title(f"Consumo por sector — CP {cp} ({nombre})", fontweight="bold")
    ax.set_ylabel("MWh")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

Observaciones iniciales del consumo por sector. El 08038 Montjuïc / Zona Franca presenta un perfil industrial visible y sostenido; el pico de 2020 probablemente se asocia a actividad que no se detuvo durante el confinamiento. En el 08005 El Poblenou los servicios dominan sobre la industria y el residencial, reflejo de la reconversión del distrito hacia la tecnología y las oficinas. El 08032 El Carmel muestra un balance entre residencial y servicios con industria casi nula y una aparente estacionalidad de verano muy marcada. En el 08002 Barri Gòtic los servicios son aplastantes; el desplome de 2020–2021 es el más pronunciado de los 4 CPs, señal directa del impacto del COVID en el turismo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=False)
fig.suptitle("Estacionalidad anual — consumo medio mensual por CP", fontweight="bold")

meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

for ax, (cp, nombre) in zip(axes.flatten(), CPS.items()):
    mensual = (
        df.filter(
            (pl.col("cod_postal") == cp) &
            (pl.col("anio") < 2026)
        )
        .group_by("mes")
        .agg(pl.col("mwh_total").mean().alias("mwh_medio"))
        .sort("mes")
    )

    ax.bar(mensual["mes"].to_numpy(), mensual["mwh_medio"].to_numpy(),
           color="#2A9D8F", alpha=0.85)
    ax.set_title(f"{cp} — {nombre}", fontsize=9, fontweight="bold")
    ax.set_ylabel("MWh medio")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(meses, fontsize=8)

plt.tight_layout()
plt.show()

Los 4 CPs muestran patrones estacionales distintos según su perfil de consumo. El industrial (08038) y el turístico (08002) pican en julio-agosto, con 195k y 155k MWh respectivamente, y caen en marzo-abril, un patrón consistente con la mayor actividad de temporada y la demanda de climatización. El residencial (08032) invierte el patrón: picos en enero y diciembre (65k MWh) y mínimos en abril-mayo (48k MWh), lo que sugiere una mayor dependencia de la calefacción que de la refrigeración. El mixto (08005) se mantiene más estable a lo largo del año, con un pico moderado en julio-agosto (150k MWh) y un segundo repunte en enero por el consumo invernal. En magnitud absoluta, Zona Franca lidera con diferencia, mientras que El Carmel se mantiene en el rango más bajo al carecer de actividad industrial o turística significativa.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=False)
fig.suptitle("Patrón semanal — consumo medio por día y hora", fontweight="bold")

dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

for ax, (cp, nombre) in zip(axes.flatten(), CPS.items()):
    semanal = (
        df.filter(pl.col("cod_postal") == cp)
        .group_by(["dia_semana", "hora"])
        .agg(pl.col("mwh_total").mean().alias("mwh_medio"))
        .sort(["dia_semana", "hora"])
    )

    for hora_val, color in zip([0, 6, 12, 18], ["#264653", "#2A9D8F", "#E9C46A", "#E76F51"]):
        subset = semanal.filter(pl.col("hora") == hora_val)
        ax.plot(subset["dia_semana"].to_numpy(),
                subset["mwh_medio"].to_numpy(),
                marker="o", label=f"{hora_val}h", color=color)

    ax.set_title(f"{cp} — {nombre}", fontsize=9, fontweight="bold")
    ax.set_ylabel("MWh medio")
    ax.set_xticks(range(7))
    ax.set_xticklabels(dias, fontsize=8)
    ax.legend(fontsize=7)
    ax.axvspan(4.5, 6.5, alpha=0.08, color="gray", label="Fin de semana")

plt.tight_layout()
plt.show()

El patrón semanal revela diferencias claras entre perfiles. El industrial, Montjuïc / Zona Franca (08038), registra la caída más pronunciada en fin de semana, con descensos de 60k MWh en los bloques de 6h y 12h del domingo respecto a los días laborables, señal directa de la paralización de la actividad industrial. El de servicios, Barri Gòtic (08002), también cae en domingo pero mantiene elevado el bloque de 18h, coherente con la actividad de servicios y ocio nocturno que no para el fin de semana. El residencial-servicios, El Carmel (08032), es el más estable a lo largo de la semana: la diferencia entre martes y domingo es mínima (unos 5 MWh), lo que confirma que el consumo residencial no responde al calendario laboral. El industria-residencial, El Poblenou (08005), presenta un comportamiento intermedio con una caída ligera en fin de semana. En cuanto a magnitudes, Montjuïc / Zona Franca lidera con bloques de 12h que superan los 200k MWh en días laborables, seguido del Barri Gòtic (165k MWh) y de Poblenou (158k MWh); El Carmel se mantiene muy por debajo (69k MWh en el bloque de 18h), reflejo de su perfil mayoritariamente residencial. En todos los CPs el bloque de 12h concentra el mayor consumo y el de 00h el mínimo, lo que confirma el patrón intradiario como una de las señales más predictivas del dataset.

Conclusiones de la descomposición de consumo. Los patrones temporales confirman que el consumo eléctrico en Barcelona responde a tres ciclos superpuestos. Estos ciclos son consistentes y repetibles, lo que valida la viabilidad del forecasting: si el consumo fuera aleatorio, predecir sería imposible; la señal estructural existe y es suficientemente fuerte. La diferencia de comportamiento entre perfiles (industrial, residencial, turístico) confirma que cod_postal y la composición sectorial son features clave para el modelo, ya que sin ellas el modelo confundiría patrones de barrios con dinámicas completamente distintas. Las variables de calendario (hora, dia_semana, mes, es_finde, es_festivo) probablemente sean features de alta importancia esperada para el modelado.

### <font color='#C0392B'><b>3.5 Descomposición STL</b></font>


In [ ]:
serie_global_stl = (
    df.group_by("datetime")
    .agg(pl.col("mwh_total").sum().alias("mwh_total"))
    .sort("datetime")
)

fechas     = serie_global_stl["datetime"].to_numpy()
serie_vals = serie_global_stl["mwh_total"].to_numpy()

res = STL(serie_vals, period=28, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
fig.suptitle("Descomposición STL — Barcelona global (todos los CPs)", fontweight="bold")

for ax, (nombre_comp, vals) in zip(axes, {
    "Serie original": serie_vals,
    "Tendencia":      res.trend,
    "Estacionalidad": res.seasonal,
    "Residuo":        res.resid,
}.items()):
    ax.plot(fechas, vals, color="#264653", linewidth=0.6)
    ax.set_ylabel(nombre_comp, fontsize=9)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

El componente de tendencia no muestra una dirección sostenida de largo plazo. Oscila entre 3M y 6M MWh, con ciclos anuales visibles pero sin crecimiento ni decrecimiento sistemático, lo que indica que la serie es estacionaria en media. El componente estacional captura el ciclo semanal (period=28 bloques) con una amplitud estable a lo largo de todo el período, oscilando entre -2M y +2M MWh, una amplitud constante e independiente del nivel de la serie. El residuo se mantiene cercano a 0 con picos puntuales que pueden corresponder a eventos no recurrentes (olas de calor, festivos, etc.).

La serie parece tener un patrón de tipo 2, estacionaria en media con una estacionalidad fuerte y regular, lo que valida el uso de SARIMA como baseline y la inclusión de variables de calendario como features principales del modelo.

#### <font color='#2A9D8F'><b>3.5.1 STL — Barri Gòtic</b></font>

In [ ]:
cp = "08002"
serie_cp = (
    df.filter(pl.col("cod_postal") == cp)
    .sort("datetime")
    .select(["datetime", "mwh_total"])
)

fechas     = serie_cp["datetime"].to_numpy()
serie_vals = serie_cp["mwh_total"].to_numpy()

In [ ]:
print(
    df.filter(
        (pl.col("cod_postal") == "08002") &
        (pl.col("datetime") >= pl.datetime(2025, 7, 1)) &
        (pl.col("datetime") < pl.datetime(2025, 11, 1))
    )
    .select(["datetime", "mwh_total", "mwh_industria", "mwh_residencial", "mwh_servicios"])
    .sort("mwh_total")
    .head(20)
)

In [ ]:
res = STL(serie_vals, period=28, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
fig.suptitle(f"Descomposición STL — CP {cp} ({CPS[cp]})", fontweight="bold")

for ax, (nombre_comp, vals) in zip(axes, {
    "Serie original": serie_vals,
    "Tendencia":      res.trend,
    "Estacionalidad": res.seasonal,
    "Residuo":        res.resid,
}.items()):
    ax.plot(fechas, vals, color="#264653", linewidth=0.6)
    ax.set_ylabel(nombre_comp, fontsize=9)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

La descomposición STL del Barri Gòtic (08002) revela una serie con estructura predecible pero con picos externos importantes. La tendencia muestra el impacto del COVID más pronunciado de los 4 CPs analizados: una caída desde 200k hasta 100k MWh entre enero y abril de 2020, reflejo del colapso del turismo y la actividad comercial. La recuperación fue gradual y no se completó hasta 2023.

La estacionalidad capturada corresponde al ciclo semanal (period=28). El ciclo anual queda absorbido en la tendencia como ondulaciones lentas recurrentes. En el Barri Gòtic, al ser un perfil de servicios/turístico, el pico anual se concentra en verano y no tiene un repunte invernal significativo. Este comportamiento contrasta con lo esperado en perfiles residenciales, donde la tendencia debería mostrar picos en enero-diciembre por la calefacción.

La estacionalidad semanal es clara y consistente, con amplitud variable, más fuerte en verano (oscilando 50k MWh) cuando la actividad turística intensifica los contrastes entre bloques horarios. El residuo se mantiene cercano a 0 la mayor parte del tiempo, con picos puntuales en 2022-2024 atribuibles a eventos no recurrentes (tal vez conciertos o festivales).

#### <font color='#2A9D8F'><b>3.5.2 STL — resto de barrios</b></font>

In [ ]:
for cp, nombre in {k: v for k, v in CPS.items() if k != "08002"}.items():
    serie_cp = (
        df.filter(pl.col("cod_postal") == cp)
        .sort("datetime")
        .select(["datetime", "mwh_total"])
    )

    fechas     = serie_cp["datetime"].to_numpy()
    serie_vals = serie_cp["mwh_total"].to_numpy()

    res = STL(serie_vals, period=28, robust=True).fit()

    fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
    fig.suptitle(f"Descomposición STL — CP {cp} ({nombre})", fontweight="bold")

    for ax, (nombre_comp, vals) in zip(axes, {
        "Serie original": serie_vals,
        "Tendencia":      res.trend,
        "Estacionalidad": res.seasonal,
        "Residuo":        res.resid,
    }.items()):
        ax.plot(fechas, vals, color="#264653", linewidth=0.6)
        ax.set_ylabel(nombre_comp, fontsize=9)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Barcelona global (todos los CPs). La tendencia no muestra una dirección sostenida de largo plazo: oscila entre 3M y 6M MWh con ciclos anuales pero sin crecimiento ni decrecimiento sistemático. La amplitud estacional es constante a lo largo de todo el período, lo que confirma que el modelo aditivo es apropiado. El residuo se mantiene cercano a 0 salvo picos puntuales atribuibles a eventos no recurrentes.

08002 Barri Gòtic (servicios/turístico). La tendencia presenta la caída COVID más pronunciada de los 4 CPs: de 200k baja a 80k MWh, reflejo del colapso del turismo y la actividad comercial. La estacionalidad se reduce durante el confinamiento y se recupera gradualmente hasta 2022. El residuo es limpio salvo picos puntuales en 2022-2024.

08038 Zona Franca (industrial). La tendencia es inestable, con un pico anómalo en noviembre de 2019 y una caída COVID moderada. La amplitud de la estacionalidad crece notablemente a partir de 2024-2025, señal de una mayor variabilidad en la actividad industrial reciente.

### <font color='#C0392B'><b>3.6 Estacionariedad: ADF y KPSS</b></font>

ADF (Augmented Dickey-Fuller) detecta si la serie tiene raíz unitaria, es decir, una tendencia que no vuelve a un valor fijo; su hipótesis nula es que la serie no es estacionaria. KPSS (Kwiatkowski-Phillips-Schmidt-Shin) plantea lo contrario: su hipótesis nula es que la serie sí es estacionaria. Se usan juntos porque se contradicen: si ambos coinciden la conclusión es clara, y si no coinciden hay estacionariedad parcial.

In [ ]:
from arch.unitroot import ADF, KPSS

CPS_ADF = {
    "08038": "Montjuïc / Zona Franca (Industrial)",
    "08005": "El Poblenou (Mixto)",
    "08032": "El Carmel / El Guinardó (Residencial)",
    "08002": "Barri Gòtic (Servicios/Turístico)",
}

resultados = []

serie_global = (
    df.group_by("datetime")
    .agg(pl.col("mwh_total").sum())
    .sort("datetime")["mwh_total"]
    .to_numpy()
)

adf_g  = ADF(serie_global, method="aic")
kpss_g = KPSS(serie_global)

resultados.append({
    "CP": "Global",
    "Barrio": "Barcelona (todos los CPs)",
    "ADF stat": round(adf_g.stat, 3),
    "ADF p-valor": round(adf_g.pvalue, 4),
    "ADF estacionaria": "Si" if adf_g.pvalue < 0.05 else "No",
    "KPSS stat": round(kpss_g.stat, 3),
    "KPSS p-valor": round(kpss_g.pvalue, 4),
    "KPSS estacionaria": "Si" if kpss_g.pvalue > 0.05 else "No",
})

for cp, nombre in CPS_ADF.items():
    serie = (
        df.filter(pl.col("cod_postal") == cp)
        .sort("datetime")["mwh_total"]
        .to_numpy()
    )

    adf   = ADF(serie, method="aic")
    kpss_ = KPSS(serie)

    resultados.append({
        "CP": cp,
        "Barrio": nombre,
        "ADF stat": round(adf.stat, 3),
        "ADF p-valor": round(adf.pvalue, 4),
        "ADF estacionaria": "Si" if adf.pvalue < 0.05 else "No",
        "KPSS stat": round(kpss_.stat, 3),
        "KPSS p-valor": round(kpss_.pvalue, 4),
        "KPSS estacionaria": "Si" if kpss_.pvalue > 0.05 else "No",
    })

pd.DataFrame(resultados)

La serie es parcialmente estacionaria: estacionaria en media pero no en varianza.

En cuanto a la estacionariedad en media (ADF), el test rechaza la hipótesis nula en los 5 casos (p < 0.05). Las series son estacionarias en media: a pesar de eventos como el COVID o la estacionalidad anual, el consumo siempre orbita alrededor de un nivel central estable, sin una tendencia sostenida.

En cuanto a la estacionariedad en varianza (KPSS), el test rechaza la hipótesis nula en los 5 casos (p < 0.05). La varianza no es constante: los cambios del fenómeno visibles en el STL (caída COVID 2020, recuperación gradual, mayor variabilidad industrial en 2024-2025) explican por qué el tamaño de los saltos cambia entre períodos.

Ambos resultados son coherentes entre sí: las series son estacionarias en media con cambios estructurales en varianza. Para SARIMA esto implica que se requerirá diferenciación estacional pero no diferenciación regular. XGBoost y LSTM/GRU son robustos a esta condición por naturaleza y no requieren transformación previa.

### <font color='#C0392B'><b>3.7 Análisis ACF y PACF</b></font>

In [ ]:
serie = (
    df.group_by("datetime")
    .agg(pl.col("mwh_total").sum())
    .sort("datetime")["mwh_total"]
    .to_numpy()
)

def acf_manual(x, nlags=112):
    n = len(x)
    x = x - x.mean()
    acf_vals = [np.dot(x[:n-k], x[k:]) / np.dot(x, x) for k in range(nlags+1)]
    return np.array(acf_vals)

def pacf_manual(x, nlags=112):
    acf_vals = acf_manual(x, nlags)
    pacf_vals = [1.0]
    for k in range(1, nlags+1):
        mat = np.array([[acf_vals[abs(i-j)] for j in range(k)] for i in range(k)])
        vec = acf_vals[1:k+1]
        try:
            coef = np.linalg.solve(mat, vec)
            pacf_vals.append(coef[-1])
        except:
            pacf_vals.append(0)
    return np.array(pacf_vals)

lags      = np.arange(113)
acf_vals  = acf_manual(serie, nlags=112)
pacf_vals = pacf_manual(serie, nlags=112)
ci        = 1.96 / np.sqrt(len(serie))

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].vlines(lags, 0, acf_vals, color="#1B3A5C", linewidth=0.8)
axes[0].axhline(0,   color="black", linewidth=0.5)
axes[0].axhline(ci,  color="gray",  linestyle="--", linewidth=0.8)
axes[0].axhline(-ci, color="gray",  linestyle="--", linewidth=0.8)
axes[0].axvline(x=4,  color="#E76F51", linestyle="--", alpha=0.7, label="lag 4 = 24h")
axes[0].axvline(x=28, color="#2A9D8F", linestyle="--", alpha=0.7, label="lag 28 = 7 dias")
axes[0].set_title("ACF — Barcelona Global", fontweight="bold")
axes[0].set_xlabel("Lag (bloques de 6h)")
axes[0].legend()

axes[1].vlines(lags, 0, pacf_vals, color="#1B3A5C", linewidth=0.8)
axes[1].axhline(0,   color="black", linewidth=0.5)
axes[1].axhline(ci,  color="gray",  linestyle="--", linewidth=0.8)
axes[1].axhline(-ci, color="gray",  linestyle="--", linewidth=0.8)
axes[1].axvline(x=4,  color="#E76F51", linestyle="--", alpha=0.7, label="lag 4 = 24h")
axes[1].axvline(x=28, color="#2A9D8F", linestyle="--", alpha=0.7, label="lag 28 = 7 dias")
axes[1].set_title("PACF — Barcelona Global", fontweight="bold")
axes[1].set_xlabel("Lag (bloques de 6h)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'lag':>5} │ {'ACF':>8} │ {'PACF':>8}")
print("──────┼──────────┼──────────")
for i in range(30):
    print(f"  {i:3d} │ {acf_vals[i]:>8.4f} │ {pacf_vals[i]:>8.4f}")

ACF (autocorrelación). Mide qué tan parecido es el consumo de hoy al de hace X bloques, acumulando todo el efecto cadena entre los lags intermedios. El lag 4 (24h, 0.83) indica que el consumo de hace 24h es muy parecido al actual: el mismo bloque horario del día anterior explica gran parte de la variabilidad. El lag 28 (7 días, 0.87) es el más parecido de toda la serie, lo que muestra que el patrón semanal domina sobre el diario. Los lags 8, 12, 16, 20 y 24 (entre 0.70 y 0.78) son picos repetidos cada 4 lags que confirman que el ciclo diario de 24h persiste a lo largo de toda la semana. El lag 2 (12h, -0.05) presenta una correlación ligeramente negativa: el bloque de hace 12h suele ser el opuesto al actual (si ahora es mediodía, hace 12h era madrugada). Los lags 56 y 84 son ecos del ciclo semanal (el mismo bloque hace 2 y 3 semanas), cuya similitud se mantiene significativa pero decrece progresivamente.

PACF (autocorrelación parcial). Mide cuánto explica el consumo de hace X bloques sobre el de hoy, una vez descontado lo que ya explican todos los lags anteriores; es la contribución directa e independiente de cada lag. El lag 1 (6h, 0.37) es una contribución directa moderada del bloque inmediatamente anterior. El lag 2 (12h, -0.22) es una contribución directa negativa: el bloque de hace 12h tiene un efecto inverso propio, consistente con el contraste entre madrugada y mediodía. El lag 3 (18h, 0.47) es una contribución directa relevante: el bloque de hace 18h aporta información propia más allá de los lags anteriores. El lag 4 (24h, 0.75) es el más alto de todos: el mismo bloque del día anterior tiene la contribución directa más fuerte, independiente del efecto cadena. El lag 5 (30h, -0.65) es una corrección negativa fuerte tras el pico del lag 4, con la que el modelo ajusta hacia abajo para no sobreestimar el efecto del día anterior. El lag 29 (7 días + 6h, -0.40) es un ajuste del ciclo semanal justo al completar la semana. A partir del lag 6 hasta el lag 28 las contribuciones son moderadas o ruido.

Como conclusión, los lags más informativos para el modelo son lag_1 (6h antes), lag_4 (24h antes, el mismo bloque del día anterior) y lag_28 (7 días antes, el mismo bloque de la semana anterior). Los lags 8 (2 días), 12 (3 días) y 16 (4 días) de la ACF son en gran parte efecto acumulado del ciclo diario, no contribuciones directas independientes, por eso no aparecen destacados en la PACF. lag_3 (18h antes) y lag_5 (30h antes) aportan una contribución directa relevante que se considerará en la configuración de SARIMA (parámetros AR y MA).

Implicaciones para feature engineering. Entran como features lag_1 (6h antes), con contribución directa confirmada por la PACF (0.37), ya que el bloque inmediatamente anterior siempre aporta señal propia; lag_4 (24h antes), el más importante de todos (PACF 0.75), la contribución directa más fuerte de la serie; lag_28 (7 días antes), con ACF 0.87, el más parecido de toda la serie, que captura el patrón semanal completo; lag_56 (2 semanas), un eco quincenal con autocorrelación aún apreciable en la ACF, que se conserva como contexto temporal de medio plazo y complementa al patrón semanal de lag_28; y rolling_mean_7d, el promedio de los últimos 7 días, que suaviza la señal y captura la tendencia reciente sin depender de un solo bloque puntual.

No entran como features lag_8, lag_12 y lag_16 (2, 3 y 4 días), que la PACF descarta porque lo que parecen aportar ya está explicado por lag_4 y lag_1 encadenados, por lo que serían pura redundancia; lag_3 y lag_5 (18h y 30h), que tienen contribución directa en la PACF pero son artefactos del ciclo diario (para SARIMA sí son relevantes como parámetros AR, pero para XGBoost y LSTM no aportan nada que lag_1 y lag_4 juntos no capturen ya); y lag_84 (3 semanas), cuya correlación decrece ya demasiado y que lag_28 ya cubre, por lo que se descarta por coste computacional sin ganancia clara.

---
# <font color='#1B3A5C'>  **4. Análisis Descriptivo de Variables Numéricas** </font>

In [ ]:
VARS_NUM = ['mwh_industria', 'mwh_residencial', 'mwh_servicios', 'mwh_no_especificado',
            'lst_celsius', 'temp_mean', 'temp_max', 'temp_min',
            'humedad_mean', 'viento_mean', 'precipitacion_sum', 'irradiancia_mean']

df_num = df.select(VARS_NUM)
df_num.head(6)

In [ ]:
df_num.describe()

Sectores de consumo. En mwh_industria la desviación estándar (19k) es mayor que la media (8.7k), y el máximo (434k) es 50 veces la mediana (1.7k), lo que refleja la heterogeneidad entre barrios: Zona Franca frente a barrios sin actividad industrial. mwh_residencial y mwh_servicios son más simétricas, ya que la media y la mediana son parecidas, y servicios domina en magnitud media (56k). mwh_no_especificado tiene un 85.35% de nulos, con un recuento real de solo 62k de 424k registros.

Temperatura. temp_mean tiene un rango de -0.26°C a 36°C, media de 17.8°C y mediana de 17.4°C, una distribución casi simétrica y sin anomalías. temp_max y temp_min pueden ser colineales con temp_mean, por lo que son candidatas a descartar en feature engineering. lst_celsius tiene una media de 24.7°C frente a los 17.9°C de temp_mean, una diferencia de aproximadamente 7°C esperada por el efecto Urban Heat Island (el asfalto y los tejados están siempre más calientes que el aire); no son redundantes porque capturan señales distintas.

Variables problemáticas. En precipitacion_sum la mediana y el P75 son 0.0: más del 75% de los bloques no tienen lluvia, por lo que es más útil como variable binaria (llueve / no llueve). irradiancia_mean tiene un mínimo de -4.27 W/m², un error de sensor, ya que es físicamente imposible, y se debe topar a 0 en feature engineering. humedad_mean y viento_mean no presentan anomalías destacables.

In [ ]:
df.select(VARS_NUM + ['mwh_total']).null_count()

mwh_industria, mwh_residencial, mwh_servicios y mwh_total tienen 0 nulos, con la imputación completada en la limpieza. mwh_no_especificado tiene 362,256 nulos (85.35%), algo normal y esperado. temp_mean y humedad_mean tienen 28,025 nulos (6.6%), que corresponden a X2 inactiva en períodos puntuales. temp_max, temp_min y precipitacion_sum tienen 136,700 nulos (32.2%), por estaciones sin cobertura completa. viento_mean e irradiancia_mean tienen 153,400 nulos (36.1%), porque X2 no tiene sensores para estas variables en todo el período. lst_celsius tiene 169,044 nulos (39.8%) por la cobertura de nubes (MODIS satelital), algo esperado. Todos los nulos restantes son estructurales y conocidos, y se mantienen para feature engineering.

Sobre estos nulos estructurales conocidos, los candidatos a imputación son temp_mean y humedad_mean (28,025 nulos, 6.6%), ya que tienen cobertura histórica completa 2019-2023 y solo fallan en X2 durante 2024-2025. El resto (viento, irradiancia, precipitación y lst_celsius) se evaluará durante el análisis bivariante para decidir si su variación entre barrios justifica imputarlos o si se gestionan directamente en feature engineering.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

VARS_NUM_PLOT = ['mwh_industria', 'mwh_residencial', 'mwh_servicios',
                 'lst_celsius', 'temp_mean', 'temp_max', 'temp_min',
                 'humedad_mean', 'viento_mean', 'precipitacion_sum', 'irradiancia_mean']

n_cols = 3
n_rows = int(np.ceil(len(VARS_NUM_PLOT) / n_cols))

fig = plt.figure(figsize=(16, n_rows * 5))
fig.suptitle('Distribución de variables numéricas', fontsize=13, fontweight='bold', y=1.02)

for i, var in enumerate(VARS_NUM_PLOT):
    v = df[var].drop_nulls().to_numpy()
    
    ax1 = fig.add_subplot(n_rows * 2, n_cols, (i // n_cols) * n_cols * 2 + (i % n_cols) + 1)
    ax1.hist(v, bins=30, color='#264653', edgecolor='white', alpha=0.85)
    ax1.set_title(var, fontsize=9, fontweight='bold')
    ax1.set_ylabel('Frecuencia', fontsize=7)
    ax1.tick_params(axis='x', rotation=45, labelsize=7)
    
    ax2 = fig.add_subplot(n_rows * 2, n_cols, (i // n_cols) * n_cols * 2 + n_cols + (i % n_cols) + 1)
    ax2.boxplot(v, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#264653', alpha=0.7),
                medianprops=dict(color='#E9C46A', linewidth=2),
                flierprops=dict(marker='o', markersize=1.5, alpha=0.3))
    ax2.tick_params(axis='x', rotation=45, labelsize=7)

plt.tight_layout()
plt.show()

mwh_industria, mwh_residencial y mwh_servicios presentan todos un sesgo positivo, producto de la heterogeneidad entre barrios y de los picos energéticos puntuales que arrastran la cola derecha. lst_celsius, temp_mean, temp_max y temp_min muestran distribuciones aparentemente bimodales, con dos picos visibles que pueden explicarse por diferencias entre barrios o por la estacionalidad (invierno frente a verano), o por ambos efectos superpuestos; se revisará en el análisis bivariante por CP y por mes. humedad_mean es la más cercana a la normal de todas las variables, con un leve sesgo negativo hacia valores altos de humedad. viento_mean tiene un sesgo positivo claro, con la mayoría de los bloques con viento bajo y una cola derecha de eventos puntuales de viento fuerte. precipitacion_sum está casi todo en 0 por el clima mediterráneo, ya que en Barcelona simplemente no llueve la mayoría de los días; el boxplot revela que, cuando sí llueve, los valores son dispersos y variables. irradiancia_mean tiene una gran masa en 0 por los bloques nocturnos (00h y 06h), ceros reales y esperados, no imputados, y el resto se distribuye ampliamente hasta 917 W/m², con un comportamiento bimodal día/noche.

Test de normalidad.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for i, var in enumerate(VARS_NUM_PLOT):
    v = df[var].drop_nulls().to_numpy()
    stats.probplot(v, dist='norm', plot=axes[i])
    axes[i].get_lines()[0].set(color='#264653', alpha=0.4, markersize=2)
    axes[i].get_lines()[1].set(color='#E76F51', linewidth=2)
    axes[i].set_title(var, fontsize=9, fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('QQ-plots — variables numéricas', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Ninguna variable sigue una distribución normal, lo que confirma la desviación sistemática de la línea de referencia en todos los QQ-plots. mwh_industria, mwh_residencial y mwh_servicios presentan curvas en S pronunciadas con colas derechas muy pesadas, el sesgo positivo extremo ya visto en los histogramas. lst_celsius, temp_mean, temp_max y temp_min son las más cercanas a la línea pero muestran un ligero baile en forma de S suave, consistente con la bimodalidad observada en los histogramas (mezcla de estaciones del año y diferencias entre barrios). humedad_mean es la más cercana a la normal visualmente, pero con una leve desviación en las colas que confirma el sesgo izquierdo ya identificado. viento_mean tiene una cola derecha clara, con los eventos extremos de viento alejándose significativamente de la línea. precipitacion_sum forma un escalón casi vertical cerca de 0, reflejo directo de la masa de ceros, y no tiene una distribución continua real. irradiancia_mean presenta un escalón pronunciado en 0 por los bloques nocturnos, seguido de una distribución dispersa, y su forma en S invertida confirma la bimodalidad día/noche.

En conclusión, se descarta Pearson para las correlaciones y se usará Spearman. Para los tests de hipótesis se emplearán Kruskal-Wallis y Mann-Whitney en lugar de ANOVA.

### <font color='#C0392B'><b>4.1 Análisis Bivariante</b></font>

Los sectores mwh_industria, mwh_residencial y mwh_servicios se excluyen del análisis bivariante por ser componentes directos de mwh_total, ya que su separación Q1/Q3 sería trivial. Se añaden temp_max y temp_min para completar el análisis meteorológico.

In [ ]:
VARS_BIN = ['temp_mean', 'lst_celsius', 'humedad_mean',
            'irradiancia_mean', 'viento_mean', 'precipitacion_sum']

ETIQUETAS = {
    'temp_mean':         ('Temperatura media del aire (°C)',    C1),
    'lst_celsius':       ('LST — Temperatura superficial (°C)', C2),
    'humedad_mean':      ('Humedad relativa (%)',                C4),
    'irradiancia_mean':  ('Irradiancia solar media (W/m²)',     C5),
    'viento_mean':       ('Velocidad media del viento (m/s)',    C3),
    'precipitacion_sum': ('Precipitacion acumulada (mm)',        C6),
}

UMBRAL = {
    'temp_mean': 50, 'lst_celsius': 50, 'humedad_mean': 50,
    'irradiancia_mean': 50, 'viento_mean': 50, 'precipitacion_sum': 30,
}

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()
fig.suptitle('Relacion entre variables meteorologicas y consumo electrico',
             fontweight='bold', fontsize=13, y=1.01)

for i, var in enumerate(VARS_BIN):
    ax     = axes[i]
    xlabel, color = ETIQUETAS[var]

    datos = (
        df.select([var, 'mwh_total'])
        .drop_nulls()
        .to_pandas()
    )

    datos['bin'] = pd.cut(datos[var], bins=25)
    resumen = (
        datos.groupby('bin', observed=True)['mwh_total']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    resumen['centro'] = resumen['bin'].apply(lambda x: x.mid)
    resumen['se']     = resumen['std'] / resumen['count'].pow(0.5)
    resumen['ci']     = 1.96 * resumen['se']
    resumen = resumen[resumen['count'] > UMBRAL[var]]

    norm = plt.Normalize(resumen['mean'].min(), resumen['mean'].max())
    cmap = plt.cm.YlOrRd

    for _, row in resumen.iterrows():
        ax.bar(row['centro'],
               row['mean'],
               width=(resumen['centro'].iloc[1] - resumen['centro'].iloc[0]) * 0.85,
               color=cmap(norm(row['mean'])),
               alpha=0.25,
               zorder=1)

    ax.plot(resumen['centro'], resumen['mean'],
            color=color, linewidth=2.5,
            marker='o', markersize=4,
            zorder=3)

    ax.scatter(resumen['centro'], resumen['mean'],
               c=resumen['mean'], cmap='YlOrRd',
               vmin=resumen['mean'].min(),
               vmax=resumen['mean'].max(),
               s=40, zorder=4, edgecolors='white', linewidths=0.5)

    n_ticks   = 8
    idx_ticks = np.linspace(0, len(resumen) - 1, n_ticks, dtype=int)
    ax.set_xticks(resumen['centro'].iloc[idx_ticks])
    ax.set_xticklabels(
        [f"{resumen['centro'].iloc[j]:.1f}" for j in idx_ticks],
        fontsize=8, rotation=0
    )

    media_global = datos['mwh_total'].mean()
    ax.axhline(media_global, color='gray', linewidth=0.8,
               linestyle='--', alpha=0.6)
    ax.text(resumen['centro'].iloc[-1], media_global * 1.005,
            'media global', fontsize=7, color='gray', ha='right')

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel('mwh_total medio', fontsize=8)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k')
    )

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()
fig.suptitle('Distribucion de consumo por rango de variable meteorologica',
             fontweight='bold', fontsize=13)

for i, var in enumerate(VARS_BIN):
    ax     = axes[i]
    xlabel = ETIQUETAS[var][0]

    datos = (
        df.select([var, 'mwh_total'])
        .drop_nulls()
        .to_pandas()
    )

    datos['bin'] = pd.cut(datos[var], bins=10)
    agrupado     = datos.groupby('bin', observed=True)
    grupos       = [g['mwh_total'].values for _, g in agrupado]
    medianas     = [np.median(g) for g in grupos]
    labels       = [str(round(b.mid, 1)) for b in agrupado.groups.keys()]

    norm = plt.Normalize(min(medianas), max(medianas))
    cmap = plt.cm.YlOrRd

    bp = ax.boxplot(
        grupos, labels=labels, patch_artist=True,
        medianprops=dict(color='white', linewidth=2),
        flierprops=dict(marker='o', markersize=2, alpha=0.3,
                        markerfacecolor='gray', markeredgewidth=0),
        whiskerprops=dict(linewidth=1.2, linestyle='--', color='gray'),
        capprops=dict(linewidth=1.5, color='gray'),
        boxprops=dict(linewidth=0.5),
        widths=0.6,
    )

    for box, med in zip(bp['boxes'], medianas):
        box.set_facecolor(cmap(norm(med)))
        box.set_alpha(0.85)

    for j, med in enumerate(medianas):
        ax.scatter(j + 1, med, color='white', s=25,
                   zorder=5, edgecolors=cmap(norm(med)), linewidths=1.5)

    ax.plot(range(1, len(medianas) + 1), medianas,
            color='gray', linewidth=1, linestyle='--', alpha=0.5, zorder=3)

    media_global = datos['mwh_total'].mean()
    ax.axhline(media_global, color='gray', linewidth=0.8, linestyle=':', alpha=0.6)
    ax.text(len(grupos) + 0.1, media_global * 1.01,
            'media\nglobal', fontsize=6, color='gray', va='bottom')

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel('mwh_total', fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.tick_params(axis='x', labelsize=7, rotation=45)

plt.tight_layout()
plt.show()

Relación entre el consumo y las variables meteorológicas (media por bins). temp_mean y lst_celsius muestran una relación no lineal con el consumo: un patrón en J para temp_mean (mínimo en torno a 15°C, subida fuerte a partir de 25°C) y en U para lst_celsius (mínimo en torno a 20°C, subida en ambos extremos). humedad_mean presenta una relación inversa moderada: los días muy secos (en torno al 15%) coinciden con olas de calor y registran el consumo más alto. irradiancia_mean es prácticamente plana entre 0 y 600 W/m², con una caída en los valores extremos atribuible a los pocos registros en ese rango. viento_mean y precipitacion_sum muestran una relación débil o nula con el consumo medio, lo que confirma su utilidad limitada como variables continuas.

Dispersión por rango (boxplot). La dispersión dentro de cada rango es muy alta en todas las variables: los bigotes alcanzan 400-500k MWh independientemente del valor de la variable meteorológica. Esto confirma que las variables meteorológicas son predictores relevantes pero no suficientes por sí solos. La mediana sube progresivamente con la temperatura en temp_mean y lst_celsius, visible en el gradiente de color de las cajas, de amarillo a rojo oscuro.

In [ ]:
from scipy.ndimage import uniform_filter1d

VARS_TREND = ['temp_mean', 'lst_celsius', 'humedad_mean', 'irradiancia_mean']

COLORES_ANIO = {
    2019: '#264653', 2020: '#2A9D8F', 2021: '#8AB17D',
    2022: '#E9C46A', 2023: '#F4A261', 2024: '#E76F51', 2025: '#C1121F',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
fig.suptitle('Evolucion anual de variables meteorologicas (2019-2025)',
             fontweight='bold', fontsize=13)

for i, var in enumerate(VARS_TREND):
    ax = axes[i]

    anual = (
        df.filter(pl.col(var).is_not_null() & (pl.col('anio') < 2026))
        .group_by('anio')
        .agg([
            pl.col(var).median().alias('mediana'),
            pl.col(var).quantile(0.25).alias('q1'),
            pl.col(var).quantile(0.75).alias('q3'),
        ])
        .sort('anio')
        .to_pandas()
    )

    anios   = anual['anio'].values
    medianas = anual['mediana'].values
    colores  = [COLORES_ANIO[a] for a in anios]

    ax.fill_between(anios, anual['q1'], anual['q3'],
                    color=C1, alpha=0.08)

    ax.plot(anios, medianas, color='gray',
            linewidth=1, linestyle='--', alpha=0.4, zorder=1)

    for x, y, c in zip(anios, medianas, colores):
        ax.scatter(x, y, color=c, s=120,
                   zorder=4, edgecolors='white', linewidths=1.5)

    for x, y, c in zip(anios, medianas, colores):
        ax.text(x, y + (anual['q3'].max() - anual['q1'].min()) * 0.03,
                f'{y:.1f}', ha='center', fontsize=8,
                color=c, fontweight='bold')
        

    from numpy.polynomial import polynomial as P
    coef    = np.polyfit(anios, medianas, 1)
    y_trend = np.polyval(coef, anios)
    
    color_trend = '#C1121F' if coef[0] > 0 else '#264653'
    
    ax.plot(anios, y_trend,
            color=color_trend, linewidth=1.5,
            linestyle='-', alpha=0.5, zorder=2)

    pendiente = coef[0]
    signo     = '▲' if pendiente > 0 else '▼'
    ax.text(anios[-1], y_trend[-1],
            f'  {signo} {abs(pendiente):.2f}/año',
            fontsize=7, color=color_trend,
            va='center', alpha=0.8)

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_ylabel('Mediana anual', fontsize=8)
    ax.set_xticks(anios)
    ax.set_xticklabels([str(a) for a in anios], fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}'))

    handles = [plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=COLORES_ANIO[a], markersize=8,
               label=str(a)) for a in anios]
    ax.legend(handles=handles, fontsize=7, loc='upper right',
              ncol=2, framealpha=0.4)

plt.tight_layout()
plt.show()

Evolución anual de las variables meteorológicas (2019-2025). temp_mean muestra una tendencia ascendente de +0.31°C/año: pasó de 16.4°C en 2019 a 18.3°C en 2025, acumulando casi 2°C en el período analizado, consistente con el calentamiento urbano progresivo documentado en Barcelona. humedad_mean desciende -0.31%/año, de 67.2% en 2019 a 66.3% en 2025; la relación inversa entre temperatura y humedad es coherente físicamente con el calentamiento del aire. lst_celsius sube de manera más ligera (+0.11°C/año), con una mayor variabilidad interanual que temp_mean: el pico de 2022 (27.9°C) contrasta con la caída de 2025 (25.6°C), explicada por el sesgo de cobertura MODIS en invierno documentado anteriormente. irradiancia_mean presenta una tendencia ligeramente descendente (-0.22 W/m²/año) pero con alta variabilidad interanual. Con 7 puntos anuales las tendencias son indicativas y se documentan como contexto para la interpretación del modelo.

In [ ]:
VARS_TREND  = ['temp_mean', 'lst_celsius', 'humedad_mean', 'irradiancia_mean']
MESES_LABEL = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']

COLORES_ANIO = {
    2019: '#264653', 2020: '#2A9D8F', 2021: '#8AB17D',
    2022: '#E9C46A', 2023: '#F4A261', 2024: '#E76F51', 2025: '#C1121F',
}

ANIOS = sorted(df.filter(pl.col('anio') < 2026)['anio'].unique().to_list())

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle('Evolucion mensual de variables meteorologicas por año',
             fontweight='bold', fontsize=13)

for i, var in enumerate(VARS_TREND):
    ax = axes[i]

    m_2019 = (
        df.filter(pl.col(var).is_not_null() & (pl.col('anio') == 2019))
        .group_by('mes').agg(pl.col(var).median().alias('mediana'))
        .sort('mes').to_pandas()
    )
    m_2025 = (
        df.filter(pl.col(var).is_not_null() & (pl.col('anio') == 2025))
        .group_by('mes').agg(pl.col(var).median().alias('mediana'))
        .sort('mes').to_pandas()
    )

    meses_comunes = sorted(set(m_2019['mes']) & set(m_2025['mes']))
    m_2019_f = m_2019[m_2019['mes'].isin(meses_comunes)].sort_values('mes')
    m_2025_f = m_2025[m_2025['mes'].isin(meses_comunes)].sort_values('mes')

    y_min = np.minimum(m_2019_f['mediana'].values, m_2025_f['mediana'].values)
    y_max = np.maximum(m_2019_f['mediana'].values, m_2025_f['mediana'].values)
    sube  = m_2025_f['mediana'].values > m_2019_f['mediana'].values

    ax.fill_between(m_2019_f['mes'], y_min, y_max,
                    where=sube,
                    color='#C1121F', alpha=0.08, label='_nolegend_')
    ax.fill_between(m_2019_f['mes'], y_min, y_max,
                    where=~sube,
                    color='#264653', alpha=0.08, label='_nolegend_')

    for anio in ANIOS:
        mensual = (
            df.filter(pl.col(var).is_not_null() & (pl.col('anio') == anio))
            .group_by('mes')
            .agg(pl.col(var).median().alias('mediana'))
            .sort('mes')
            .to_pandas()
        )

        lw    = 2.5 if anio in [2019, 2025] else 1.2
        alpha = 1.0 if anio in [2019, 2025] else 0.45
        ms    = 5   if anio in [2019, 2025] else 3

        ax.plot(mensual['mes'], mensual['mediana'],
                color=COLORES_ANIO[anio],
                linewidth=lw, alpha=alpha,
                marker='o', markersize=ms,
                label=str(anio),
                zorder=5 if anio in [2019, 2025] else 2)

    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_ylabel('Mediana', fontsize=8)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MESES_LABEL, fontsize=8)

    handles = [plt.Line2D([0], [0], color=COLORES_ANIO[a],
               linewidth=2, label=str(a)) for a in ANIOS]
    ax.legend(handles=handles, fontsize=7,
              loc='upper right', ncol=2, framealpha=0.4)

plt.tight_layout()
plt.show()

Evolución mensual por año (2019-2025). temp_mean confirma la tendencia ascendente mes a mes: la línea de 2025 (rojo) se sitúa consistentemente por encima de la de 2019 (verde oscuro) en los meses de primavera e inicio de verano, con la banda roja entre ambos años visible en mayo-junio. lst_celsius muestra el mismo patrón en primavera pero con la banda azul en enero-marzo, donde 2025 está por debajo de 2019; esto es el sesgo de cobertura MODIS en invierno ya documentado, no un enfriamiento real. humedad_mean presenta una mayor variabilidad interanual que las variables térmicas, con la banda roja (2025 > 2019) en enero-febrero y la banda azul (2025 < 2019) en mayo-agosto, sin un patrón estacional claro y consistente entre años. irradiancia_mean es la más ruidosa de las cuatro: la línea de 2025 destaca con picos anómalos en mayo-junio (en torno a 150 W/m²) que no se repiten en otros años, probablemente asociados a períodos excepcionales de cielo despejado o a artefactos de cobertura MODIS. El patrón estacional de todas las variables se mantiene estable entre años (verano siempre más cálido, invierno más frío), lo que confirma que la estacionalidad es robusta y predecible para el modelo.

Temperatura y LST. temp_mean y lst_celsius muestran una distribución bimodal en el histograma univariante, y el KDE por estación confirma que son dos poblaciones superpuestas: invierno (pico en torno a 12°C) y verano (pico en torno a 25°C). En el bivariante Q1 frente a Q3, el consumo alto presenta ambos picos (frío y calor) mientras que el consumo bajo se concentra en temperaturas intermedias, lo que confirma la relación en U con mwh_total y una señal predictiva fuerte. lst_celsius aporta una señal complementaria a temp_mean, no redundante: temp_mean mide la temperatura del aire desde la estación Meteocat (igual para todos los CPs de la misma estación), mientras que lst_celsius varía por CP y captura las diferencias intraurbanas por Urban Heat Island. Ambas se mantienen en el modelo.

Humedad. Sin bimodalidad: el KDE muestra las 4 estaciones solapadas en torno al 60-75%. El bivariante Q1 frente a Q3 revela que el consumo alto se desplaza hacia humedad baja, coherente con los días calurosos y secos de verano que generan una mayor demanda de refrigeración.

Viento. Las 4 estaciones son prácticamente idénticas en el KDE, sin patrón estacional. Las distribuciones Q1 frente a Q3 se solapan casi por completo. Es la variable con menor poder discriminante del grupo.

Irradiancia. El KDE expone la causa de su bimodalidad: el pico en 0 corresponde al invierno (días cortos, bloques nocturnos), mientras que la cola hasta 800 W/m² es el verano. El bivariante Q1 frente a Q3 queda distorsionado por la masa de ceros nocturnos y requiere un análisis separado por bloque horario diurno.

Precipitación. Todas las estaciones están pegadas al 0 en el KDE, sin separación en el bivariante. Esto confirma su utilidad limitada como variable continua y la convierte en candidata a binarizar (llueve / no llueve).

Limitación MODIS LST. Se utilizó MOD11A1 (satélite Terra, paso en torno a las 10:30h local), asignando el mismo valor de LST a todos los bloques del día. El producto MYD11A1 (satélite Aqua, en torno a las 13:30h local) capturaría mejor el pico de Urban Heat Island al coincidir con el máximo de radiación solar. Ambos productos podrían combinarse asignando Terra al bloque 06h-12h y Aqua al bloque 12h-18h, obteniendo una LST específica por bloque diurno. Se documenta como línea futura.

### <font color='#C0392B'><b>4.2 Test de Hipótesis Mann-Whitney U</b></font>

In [ ]:
q1_t = df['mwh_total'].quantile(0.25)
q3_t = df['mwh_total'].quantile(0.75)

VARS_MW = ['temp_mean', 'temp_max', 'temp_min', 'humedad_mean',
           'viento_mean', 'precipitacion_sum', 'irradiancia_mean', 'lst_celsius']

resultados_mw = []
for var in VARS_MW:
    df_pair = df.select([var, 'mwh_total']).drop_nulls()
    g_bajo = df_pair.filter(pl.col('mwh_total') <= q1_t)[var].to_numpy()
    g_alto = df_pair.filter(pl.col('mwh_total') >= q3_t)[var].to_numpy()
    stat_mw, p_mw = stats.mannwhitneyu(g_bajo, g_alto, alternative='two-sided')
    resultados_mw.append({
        'variable':      var,
        'mediana_bajo':  round(float(np.median(g_bajo)), 3),
        'mediana_alto':  round(float(np.median(g_alto)), 3),
        'p_value':       p_mw,
        'significativo': 'si' if p_mw < 0.05 else 'no'
    })

res_mw = sorted(resultados_mw, key=lambda x: x['p_value'])

print(f"{'variable':<25} {'mediana_bajo':>14} {'mediana_alto':>14} {'p_value':>14} {'sig':>5}")
print('-' * 80)
for r in res_mw:
    print(f"{r['variable']:<25} {r['mediana_bajo']:>14.3f} {r['mediana_alto']:>14.3f} {r['p_value']:>14.4e} {r['significativo']:>5}")

Todas las variables son significativas (p < 0.05) pero con diferencias prácticas muy distintas entre ellas. irradiancia_mean muestra la mayor diferencia: 193 W/m² en consumo alto frente a 7.3 W/m² en consumo bajo, reflejo del patrón horario diurno más que de una relación causal directa. temp_mean y lst_celsius confirman la relación térmica: el consumo alto aparece en días más cálidos (19.0°C frente a 16.6°C, y 27.2°C frente a 24.6°C respectivamente). humedad_mean presenta una relación inversa moderada: consumo alto en días más secos (65.4% frente a 70.9%), coherente con las olas de calor. viento_mean es significativa pero la diferencia práctica es mínima (2.1 frente a 2.3 m/s), lo que cuestiona su utilidad como feature continua. precipitacion_sum tiene ambas medianas en 0: la información útil está en si llueve o no, no en la cantidad exacta.

---
# <font color='#1B3A5C'>  **5. Análisis Descriptivo de Variables Categóricas** </font>

In [ ]:
VARS_CAT = ['hora', 'dia_semana', 'mes', 'anio', 'es_finde', 'es_festivo']

for var in VARS_CAT:
    conteo = (
        df.group_by(var)
        .agg(
            pl.len().alias('n'),
            pl.col('mwh_total').mean().round(1).alias('mwh_medio')
        )
        .sort(var)
        .with_columns((pl.col('n') / len(df) * 100).round(2).alias('pct'))
    )
    print(f"\n{'='*50}")
    print(f"  {var}")
    print(f"{'='*50}")
    print(conteo)

print(f"\n{'='*50}")
print(f"  cod_postal")
print(f"{'='*50}")
print(
    df.filter(pl.col('nombre_postal').is_not_null())
    .group_by(['cod_postal', 'nombre_postal'])
    .agg([
        pl.len().alias('n'),
        pl.col('mwh_total').mean().round(1).alias('mwh_medio'),
        pl.col('mwh_total').median().round(1).alias('mwh_mediana'),
        pl.col('mwh_total').max().round(1).alias('mwh_max'),
        pl.col('mwh_industria').sum().alias('sum_ind'),
        pl.col('mwh_residencial').sum().alias('sum_res'),
        pl.col('mwh_servicios').sum().alias('sum_serv'),
    ])
    .with_columns([
        (pl.col('sum_ind') / (pl.col('sum_ind') + pl.col('sum_res') + pl.col('sum_serv')) * 100).round(1).alias('pct_ind'),
        (pl.col('sum_res') / (pl.col('sum_ind') + pl.col('sum_res') + pl.col('sum_serv')) * 100).round(1).alias('pct_res'),
        (pl.col('sum_serv') / (pl.col('sum_ind') + pl.col('sum_res') + pl.col('sum_serv')) * 100).round(1).alias('pct_serv'),
    ])
    .drop(['sum_ind', 'sum_res', 'sum_serv'])
    .sort('mwh_medio', descending=True)
)

Patrones temporales. En la hora, el valle está en el bloque de 00h (71k MWh) y el pico en el de 12h (120k MWh), una diferencia del 69%; la distribución está balanceada al 25% por bloque, lo que confirma una grilla temporal completa. En el día de la semana hay una caída progresiva hacia el fin de semana: los laborables rondan los 105-106k MWh frente a los 91k del sábado y los 86k del domingo, una diferencia de 20k MWh entre miércoles y domingo. En el mes se observa un doble pico en enero (110k MWh) y julio (118k MWh) y un valle en abril (89k MWh), lo que confirma la estacionalidad por calefacción invernal y refrigeración estival. En el año hay una caída notable en 2020 (99.7k MWh) frente a 2019 (113.3k MWh) por el COVID, con recuperación parcial en 2021-2022 y tendencia descendente desde 2023. es_finde reduce el consumo un 11% (106k frente a 94k MWh), y es_festivo lo reduce un 15% (101k frente a 86k MWh), un impacto mayor que el del fin de semana por la paralización simultánea de industria, servicios y comercio.

Código postal. El rango de consumo es extremo: desde 29.7k MWh en el 08033 Vallbona hasta 233.6k MWh en el 08040 Zona Franca, una diferencia de 8 veces entre el mínimo y el máximo. Los CPs con mayor componente industrial lideran: Zona Franca (08040, 29% ind.), Montjuïc (08038, 37% ind.) y El Bon Pastor (08030, 16% ind.). En el extremo opuesto están los barrios residenciales periféricos: El Carmel (08032, 70% res.), Vilapicina (08031, 64% res.) y Vallbona (08033, 57% res.).

In [ ]:
VARS_BINARIAS = {
    'es_finde':   {'valores': [0, 1], 'labels': ['Laborable', 'Fin de semana'],
                   'colores': [C1, C4]},
    'es_festivo': {'valores': [0, 1], 'labels': ['No festivo', 'Festivo'],
                   'colores': [C1, C5]},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Consumo electrico — laborable vs fin de semana y festivos',
             fontweight='bold', fontsize=13)

for row, (var, cfg) in enumerate(VARS_BINARIAS.items()):
    grupos  = [df.filter(pl.col(var) == v)['mwh_total'].drop_nulls().to_numpy()
               for v in cfg['valores']]
    labels  = cfg['labels']
    colores = cfg['colores']
    medias  = [g.mean() for g in grupos]
    medianas = [np.median(g) for g in grupos]

    ax_bar = axes[row, 0]
    bars = ax_bar.bar(labels, medias, color=colores, alpha=0.85,
                      edgecolor='white', linewidth=0.5, width=0.5)
    for bar, val in zip(bars, medias):
        ax_bar.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.01,
                    f'{val/1000:.1f}k', ha='center',
                    fontsize=10, fontweight='bold',
                    color=bar.get_facecolor())
    dif = (medias[0] - medias[1]) / medias[0] * 100
    ax_bar.text(0.98, 0.95, f'Reduccion: {dif:.1f}%',
                transform=ax_bar.transAxes, ha='right', va='top',
                fontsize=9, color='gray')
    ax_bar.set_title(f'Consumo medio — {var}', fontsize=10, fontweight='bold')
    ax_bar.set_ylabel('MWh medio', fontsize=8)
    ax_bar.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax_bar.set_ylim(0, max(medias) * 1.15)

    ax_box = axes[row, 1]
    bp = ax_box.boxplot(grupos, labels=labels, patch_artist=True,
                        showfliers=False,
                        medianprops=dict(color='white', linewidth=2),
                        whiskerprops=dict(linewidth=1.2, linestyle='--', color='gray'),
                        capprops=dict(linewidth=1.5, color='gray'),
                        boxprops=dict(linewidth=0.5),
                        widths=0.4)
    for patch, color in zip(bp['boxes'], colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    for j, med in enumerate(medianas):
        ax_box.scatter(j + 1, med, color='white', s=40,
                       zorder=5, edgecolors=colores[j], linewidths=1.5)

    media_global = df['mwh_total'].mean()
    ax_box.axhline(media_global, color='gray', linewidth=0.8,
                   linestyle=':', alpha=0.6)
    ax_box.text(2.3, media_global * 1.01, 'media global',
                fontsize=7, color='gray')
    ax_box.set_title(f'Distribucion — {var}', fontsize=10, fontweight='bold')
    ax_box.set_ylabel('MWh', fontsize=8)
    ax_box.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

es_finde y es_festivo. El fin de semana reduce el consumo aproximadamente un 11% respecto a los laborables (94.2k frente a 106.2k MWh). Los festivos lo reducen aproximadamente un 15% (86.3k frente a 101.7k MWh), un impacto mayor que el del fin de semana porque paralizan simultáneamente industria, servicios y comercio, mientras que el sábado mantiene parte de la actividad de servicios. El boxplot confirma que la reducción no es solo en la media, sino en toda la distribución: el IQR del fin de semana y de los festivos se desplaza hacia abajo, con medianas en torno a 90k y 86k MWh respectivamente. La dispersión es similar en todos los grupos, por lo que la variabilidad no cambia según el tipo de día, solo baja el nivel general de consumo.

In [ ]:
CPS_ESTACIONAL = {
    "08040": "Zona Franca",
    "08038": "Montjuïc",
    "08002": "Barri Gòtic",
    "08036": "Antiga Esquerra",
    "08032": "El Carmel",
    "08033": "Vallbona",
}

COLORES_ESTACIONAL = {
    "08040": C1,
    "08038": C2,
    "08002": C3,
    "08036": C4,
    "08032": C5,
    "08033": C6,
}

meses_label = ["Ene","Feb","Mar","Abr","May","Jun",
                "Jul","Ago","Sep","Oct","Nov","Dic"]

df_sel  = df.filter(pl.col("cod_postal").is_in(list(CPS_ESTACIONAL.keys())))
mensual = (
    df_sel
    .group_by(["cod_postal", "mes"])
    .agg(pl.col("mwh_total").mean().alias("mwh_medio"))
    .sort(["cod_postal", "mes"])
)

fig, ax = plt.subplots(figsize=(14, 6))

for cp, nombre in CPS_ESTACIONAL.items():
    serie = (
        mensual.filter(pl.col("cod_postal") == cp)
        .sort("mes")
    )
    meses = serie["mes"].to_list()
    vals  = serie["mwh_medio"].to_list()

    ax.plot(meses, vals,
            color=COLORES_ESTACIONAL[cp],
            marker='o', markersize=5,
            linewidth=2)

    ax.text(12.15, vals[-1], f'{cp} · {nombre}',
            fontsize=7.5, color=COLORES_ESTACIONAL[cp],
            va='center', fontweight='bold')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_label, fontsize=9)
ax.set_ylabel('MWh medio por bloque de 6h', fontsize=9)
ax.set_title('Consumo mensual medio por barrio (2019-2025)',
             fontweight='bold', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.set_xlim(1, 14.5)

plt.tight_layout()
plt.show()

Consumo mensual medio por barrio (6 perfiles). Los barrios industriales (08040 Zona Franca, 08038 Montjuïc) mantienen un consumo alto y estable todo el año con un leve pico invernal, ya que la producción continua amortigua la estacionalidad. Los barrios de servicios (08002 Barri Gòtic, 08036 Antiga Esquerra) muestran el pico de julio más pronunciado y una caída marcada en abril-mayo, un patrón típico de climatización estival. Los residenciales (08032 El Carmel, 08033 Vallbona) siguen la curva en U con picos en enero y julio, pero con una amplitud mucho menor por su bajo volumen absoluto. La separación entre Zona Franca (en torno a 270k MWh en julio) y Vallbona (en torno a 30k MWh) reproduce la heterogeneidad de 8 veces entre barrios ya documentada, lo que justifica que el modelo trate cada CP como una serie temporal independiente.

### <font color='#C0392B'><b>5.1 Varianza de barrios</b></font>


In [ ]:
varianza_cp = (
    df.group_by("cod_postal")
    .agg([
        pl.col("mwh_total").std().alias("std"),
        pl.col("mwh_total").mean().alias("media"),
        pl.col("mwh_industria").sum().alias("sum_ind"),
        pl.col("mwh_residencial").sum().alias("sum_res"),
        pl.col("mwh_servicios").sum().alias("sum_serv"),
        pl.col("nombre_postal").first(),
    ])
    .with_columns([
        (pl.col("std") / pl.col("media") * 100).round(1).alias("cv_pct"),
        (pl.col("sum_ind") / (pl.col("sum_ind") + pl.col("sum_res") + pl.col("sum_serv")) * 100).round(1).alias("pct_ind"),
        (pl.col("sum_res") / (pl.col("sum_ind") + pl.col("sum_res") + pl.col("sum_serv")) * 100).round(1).alias("pct_res"),
        (pl.col("sum_serv") / (pl.col("sum_ind") + pl.col("sum_res") + pl.col("sum_serv")) * 100).round(1).alias("pct_serv"),
    ])
    .select(["cod_postal", "nombre_postal", "media", "std", "cv_pct", "pct_ind", "pct_res", "pct_serv"])
    .sort("cv_pct", descending=True)
)
print(varianza_cp)

In [ ]:
vcp_pd = varianza_cp.to_pandas()

fig, ax = plt.subplots(figsize=(12, 14))

norm    = plt.Normalize(vcp_pd['cv_pct'].min(), vcp_pd['cv_pct'].max())
colores_cv = [C5 if cv > 50 else C3 if cv > 35 else C1
              for cv in vcp_pd['cv_pct']]

bars = ax.barh(vcp_pd['nombre_postal'], vcp_pd['cv_pct'],
               color=colores_cv, alpha=0.85,
               edgecolor='white', linewidth=0.3,
               height=0.7)

ax.axvline(35, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(50, color=C5,    linewidth=0.8, linestyle='--', alpha=0.5)
ax.text(35.5, 41, 'CV=35%', fontsize=7, color='gray')
ax.text(50.5, 41, 'CV=50%', fontsize=7, color=C5)

for bar, val in zip(bars, vcp_pd['cv_pct']):
    color_txt = C5 if val > 50 else C3 if val > 35 else C1
    ax.text(bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=7,
            color=color_txt, fontweight='bold')

ax.set_xlabel('Coeficiente de variacion (%)', fontsize=9)
ax.set_title('Variabilidad de consumo por barrio (CV = std/media)',
             fontweight='bold', fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.invert_yaxis()
ax.set_xlim(0, 80)

plt.tight_layout()
plt.show()

Variabilidad de consumo por barrio (CV). Los barrios con mayor volatilidad son de servicios: Sant Antoni (CV=71.6%), L'Antiga Esquerra (CV=58.8%) y Dreta de l'Eixample (CV=54-58%). Su consumo se concentra en los bloques diurnos laborables y colapsa en las madrugadas y los festivos, lo que genera una dispersión intrínsecamente alta. Los barrios industriales y portuarios presentan los CV más bajos: La Zona Franca (CV=23.2%), El Port (CV=23.2%) y La Verneda (CV=21.8%), ya que la producción continua en turnos suaviza los picos y valles del calendario. Esta heterogeneidad tiene implicaciones directas para el modelado: los barrios con CV alto concentrarán los errores más grandes y la tasa de detección de picos será especialmente exigente en esos CPs.

### <font color='#C0392B'><b>5.2 Análisis Bivariante de Variables Categóricas</b></font>


In [ ]:
consumo_pd = (
    df.filter(pl.col('nombre_postal').is_not_null())
    .group_by(['cod_postal', 'nombre_postal'])
    .agg(pl.col('mwh_total').mean().round(1).alias('mwh_medio'))
    .sort('mwh_medio', descending=True)
    .to_pandas()
)

consumo_pd['label'] = consumo_pd['cod_postal'] + ' · ' + consumo_pd['nombre_postal']

fig, ax = plt.subplots(figsize=(14, 10))

norm    = plt.Normalize(consumo_pd['mwh_medio'].min(),
                        consumo_pd['mwh_medio'].max())
colores = [plt.cm.YlOrRd(norm(v)) for v in consumo_pd['mwh_medio']]

bars = ax.barh(consumo_pd['label'], consumo_pd['mwh_medio'],
               color=colores, alpha=0.85,
               edgecolor='white', linewidth=0.3)

for bar, val in zip(bars, consumo_pd['mwh_medio']):
    ax.text(bar.get_width() + 500,
            bar.get_y() + bar.get_height() / 2,
            f'{val/1000:.1f}k', va='center', fontsize=7.5,
            fontweight='bold', color='#333333')

ax.set_xlabel('MWh medio por bloque de 6h', fontsize=9)
ax.set_title('Consumo medio por barrio (2019-2025)',
             fontweight='bold', fontsize=12)
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k')
)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

Consumo medio por barrio (2019-2025). El rango es de 8 veces entre el mínimo y el máximo: desde 29.7k MWh en el 08033 Vallbona hasta 233.6k MWh en el 08040 La Zona Franca. Los tres CPs con mayor consumo son industriales o portuarios: Zona Franca (08040), El Bon Pastor (08030) y Zona Universitaria (08028). Los CPs con consumo bajo del centro (08010 Pl. Catalunya, 08008-08009 Dreta de l'Eixample) no reflejan una baja actividad económica, sino que son códigos postales de superficie reducida que cubren solo una fracción del barrio. La heterogeneidad de 8 veces justifica tratar cod_postal como variable categórica: un modelo sin distinción espacial promediaría perfiles completamente distintos.

In [ ]:
with open('/home/app/src/data/BARCELONA.geojson', 'r') as f:
    geojson_bcn = json.load(f)

MAPA_BASE = dict(
    mapbox_style='carto-positron',
    zoom=11,
    center={'lat': 41.3874, 'lon': 2.1686},
    opacity=0.85,
)

LAYOUT_CLARO = dict(
    title_font_size=16,
    title_font_color='#1B3A5C',
    paper_bgcolor='white',
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0},
    height=700, width=1000,
)

In [ ]:
consumo_cp = (
    df.filter(pl.col('nombre_postal').is_not_null())
    .group_by(['cod_postal', 'nombre_postal'])
    .agg(pl.col('mwh_total').mean().round(1).alias('mwh_medio'))
    .sort('cod_postal')
    .to_pandas()
)

fig = px.choropleth_mapbox(
    consumo_cp, geojson=geojson_bcn,
    locations='cod_postal', featureidkey='properties.COD_POSTAL',
    color='mwh_medio',
    color_continuous_scale='YlOrRd',
    hover_name='nombre_postal',
    hover_data={'cod_postal': True, 'mwh_medio': ':.0f'},
    labels={'mwh_medio': 'MWh medio'},
    title='Consumo medio por barrio (2019-2025, MWh por bloque 6h)',
    range_color=[consumo_cp['mwh_medio'].min(),
                 consumo_cp['mwh_medio'].quantile(0.95)],
    **MAPA_BASE
)
fig.update_layout(**LAYOUT_CLARO, coloraxis_colorbar=colorbar_claro('MWh medio'))
fig.show()

In [ ]:

C1 = '#264653'
C3 = '#E9C46A'
C5 = '#E76F51'
TITULO = '#1B3A5C'

vcp_pd = varianza_cp.to_pandas()

fig, ax = plt.subplots(figsize=(14, 16))

colores_cv = [C5 if cv > 50 else C3 if cv > 35 else C1
              for cv in vcp_pd['cv_pct']]

bars = ax.barh(vcp_pd['nombre_postal'], vcp_pd['cv_pct'],
               color=colores_cv, alpha=0.8,
               edgecolor='#cccccc', linewidth=0.5,
               height=0.65)

ax.grid(axis='x', alpha=0.3, linestyle=':', linewidth=0.5, color='gray')
ax.set_axisbelow(True)

ax.axvline(35, color='gray', linewidth=1, linestyle='--', alpha=0.5, label='CV=35% (variabilidad media)')
ax.axvline(50, color=C5, linewidth=1, linestyle='--', alpha=0.6, label='CV=50% (variabilidad alta)')

for bar, val in zip(bars, vcp_pd['cv_pct']):
    ax.text(val + 1.5,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%',
            va='center', fontsize=8,
            color=TITULO, fontweight='bold')

ax.set_xlabel('Coeficiente de variación (%)', fontsize=10, fontweight='bold', color=TITULO)
ax.set_title('Variabilidad de consumo por barrio\n(CV = std/media)', 
             fontsize=13, fontweight='bold', color=TITULO, pad=20)
ax.set_xlim(0, 85)

ax.tick_params(axis='y', labelsize=8, colors=TITULO)
ax.tick_params(axis='x', labelsize=8, colors=TITULO)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')

ax.legend(loc='lower right', fontsize=8, frameon=True, fancybox=True, shadow=True)

ax.invert_yaxis()

ax.set_facecolor('#fafafa')
fig.patch.set_facecolor('white')

plt.tight_layout()
plt.show()

In [ ]:
print(varianza_cp.group_by('nombre_postal').agg(pl.len()).filter(pl.col('len') > 1))

print(varianza_cp.sort('cv_pct', descending=True))

### <font color='#C0392B'><b>5.3 Heat Maps</b></font>


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

todos_cp = (
    df.filter(pl.col('nombre_postal').is_not_null())
    .group_by(['cod_postal', 'mes'])
    .agg(pl.col('mwh_total').mean().alias('mwh_medio'))
    .sort(['cod_postal', 'mes'])
)

cps_validos = [
    cp for cp in df['cod_postal'].unique().to_list()
    if todos_cp.filter(pl.col('cod_postal') == cp).shape[0] == 12
]

matriz = np.zeros((len(cps_validos), 12))
for i, cp in enumerate(cps_validos):
    vals = todos_cp.filter(pl.col('cod_postal') == cp).sort('mes')['mwh_medio'].to_numpy()
    vmin, vmax = vals.min(), vals.max()
    matriz[i] = (vals - vmin) / (vmax - vmin) if vmax > vmin else vals

linkage_matrix = linkage(pdist(matriz, metric='euclidean'), method='ward')
orden = dendrogram(linkage_matrix, no_plot=True)['leaves']
matriz_ord  = matriz[orden]
cps_ord     = [cps_validos[i] for i in orden]

nombres_cp = (
    df.filter(pl.col('nombre_postal').is_not_null())
    .select(['cod_postal', 'nombre_postal'])
    .unique()
    .to_pandas()
    .set_index('cod_postal')['nombre_postal']
    .to_dict()
)
labels_y    = [f"{cp} · {(nombres_cp.get(cp) or '')[:16]}" for cp in cps_ord]
meses_label = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']

fig, ax = plt.subplots(figsize=(12, 14))

im = ax.imshow(matriz_ord, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

for x in range(13):
    ax.axvline(x - 0.5, color='white', linewidth=0.8, alpha=0.6)
for y in range(len(cps_ord) + 1):
    ax.axhline(y - 0.5, color='white', linewidth=0.4, alpha=0.4)

ax.set_xticks(range(12))
ax.set_xticklabels(meses_label, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(cps_ord)))
ax.set_yticklabels(labels_y, fontsize=7.5)
ax.set_title('Patron estacional por barrio, consumo normalizado (0=min, 1=max)',
             fontweight='bold', fontsize=12, pad=12)

cbar = plt.colorbar(im, ax=ax, label='Intensidad relativa', shrink=0.35)
cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

Patrón estacional por barrio (heatmap normalizado). No existe un único patrón estacional en Barcelona: el clustering jerárquico identifica al menos dos regímenes diferenciados. Los barrios de servicios (08036 L'Antiga Esquerra, 08002 Barri Gòtic, 08013 La Sagrada Família) concentran su pico en julio, respondiendo a la refrigeración estival. Los barrios con mayor componente industrial o portuaria (08040 Zona Franca, 08038 Montjuïc, 08039 El Port) muestran su pico en enero, dominados por la calefacción y los turnos continuos de producción. Un tercer grupo (08011 Sant Antoni, 08034 Pedralbes) presenta patrones fragmentados sin un mes dominante claro, coherente con los coeficientes de variación más altos identificados en el análisis de variabilidad. Esta heterogeneidad refuerza la necesidad de tratar cod_postal como variable categórica: un modelo global promediaría regímenes opuestos y perdería la señal estacional de ambos grupos.

In [ ]:
pivot = (
    df.group_by(['dia_semana', 'hora'])
    .agg(pl.col('mwh_total').mean().alias('mwh_medio'))
    .sort(['dia_semana', 'hora'])
)

pivot_pd = pivot.to_pandas().pivot(index='dia_semana', columns='hora', values='mwh_medio')
pivot_pd.index   = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
pivot_pd.columns = ['00h', '06h', '12h', '18h']

fig, ax = plt.subplots(figsize=(9, 6))

im = ax.imshow(pivot_pd.values, aspect='auto',
               cmap='YlOrRd', vmin=pivot_pd.values.min() * 0.95)

for x in np.arange(-0.5, 4, 1):
    ax.axvline(x, color='white', linewidth=1.5, alpha=0.6)
for y in np.arange(-0.5, 7, 1):
    ax.axhline(y, color='white', linewidth=1.5, alpha=0.6)

umbral = pivot_pd.values.max() * 0.70
for i in range(7):
    for j in range(4):
        val   = pivot_pd.values[i, j]
        color = 'white' if val > umbral else '#333333'
        ax.text(j, i, f'{val/1000:.1f}k',
                ha='center', va='center',
                fontsize=10, fontweight='bold', color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('MWh medio', fontsize=9)
cbar.ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k')
)

ax.set_xticks(range(4))
ax.set_xticklabels(pivot_pd.columns, fontsize=10)
ax.set_yticks(range(7))
ax.set_yticklabels(pivot_pd.index, fontsize=10)
ax.set_title('Consumo por bloque horario y dia de semana',
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('Hora del bloque', fontsize=9)
ax.set_ylabel('Dia de la semana', fontsize=9)

plt.tight_layout()
plt.show()

Consumo por bloque horario y día de la semana. El bloque nocturno (00h) es prácticamente constante toda la semana, con un rango de 69.1k a 71.9k MWh, una diferencia inferior al 4%: la madrugada homogeneiza los patrones independientemente del tipo de día. A medida que avanza el día, la diferencia entre días laborables y fin de semana se amplifica: en el bloque de 12h la diferencia entre miércoles (128.3k MWh) y domingo (97.1k MWh) alcanza los 31k MWh, una brecha del 24% que no existía a las 00h. Este efecto de amplificación diurna es la firma del consumo industrial y de servicios, que se concentra en las horas centrales y desaparece los domingos. El sábado presenta un perfil intermedio asimétrico: el bloque de 06h cae abruptamente (89.8k MWh frente a unos 111k los laborables) mientras que el de 12h recupera parcialmente (105.4k MWh), reflejo de la actividad comercial que arranca más tarde y con menor intensidad.

El efecto del día de la semana depende del bloque horario. Esto justifica incluir hora_x_finde como feature de ingeniería, ya que un modelo con hora y dia_semana independientes no captura que el perfil horario del domingo es cualitativamente distinto al del miércoles.

### <font color='#C0392B'><b>5.4 Mapas</b></font>

In [ ]:



fig = px.choropleth_mapbox(
    consumo_anio, geojson=geojson_bcn,
    locations='cod_postal', featureidkey='properties.COD_POSTAL',
    color='mwh_medio', animation_frame='anio',
    color_continuous_scale='YlOrRd',
    hover_name='nombre_postal',
    hover_data={'cod_postal': True, 'mwh_medio': ':.0f'},
    labels={'mwh_medio': 'MWh medio'},
    title='Consumo medio por codigo postal (MWh por bloque 6h)',
    range_color=[consumo_anio['mwh_medio'].min(),
                 consumo_anio['mwh_medio'].quantile(0.95)],
    **MAPA_BASE
)
fig.update_layout(**LAYOUT_CLARO, coloraxis_colorbar=colorbar_claro('MWh medio'))
fig.show()

lst_cp = (
    df.filter(pl.col('nombre_postal').is_not_null() &
              pl.col('lst_celsius').is_not_null() &
              pl.col('anio').is_not_null())
    .group_by(['cod_postal', 'nombre_postal', 'anio'])
    .agg(pl.col('lst_celsius').mean().round(2).alias('lst_media'))
    .sort(['cod_postal', 'anio'])
    .with_columns(pl.col('anio').cast(pl.Utf8))
    .to_pandas()
)

temp_cp = (
    df.filter(pl.col('nombre_postal').is_not_null() &
              pl.col('anio').is_not_null() &
              pl.col('temp_mean').is_not_null())
    .group_by(['cod_postal', 'nombre_postal', 'anio'])
    .agg(pl.col('temp_mean').mean().round(2).alias('temp_media'))
    .sort(['cod_postal', 'anio'])
    .with_columns(pl.col('anio').cast(pl.Utf8))
    .to_pandas()
)

temp_min_global = min(temp_cp['temp_media'].min(),
                      lst_cp['lst_media'].quantile(0.10))
temp_max_global = max(temp_cp['temp_media'].quantile(0.95),
                      lst_cp['lst_media'].quantile(0.95))

fig_lst = px.choropleth_mapbox(
    lst_cp, geojson=geojson_bcn,
    locations='cod_postal', featureidkey='properties.COD_POSTAL',
    color='lst_media', animation_frame='anio',
    color_continuous_scale='RdBu_r',
    hover_name='nombre_postal',
    hover_data={'lst_media': ':.2f', 'anio': True},
    labels={'lst_media': 'LST media (°C)'},
    title='Temperatura superficial media LST MODIS por codigo postal',
    range_color=[temp_min_global, temp_max_global],
    **MAPA_BASE
)
fig_lst.update_layout(**LAYOUT_CLARO,
                      coloraxis_colorbar=colorbar_claro('LST media (°C)'))
fig_lst.show()

fig_temp = px.choropleth_mapbox(
    temp_cp, geojson=geojson_bcn,
    locations='cod_postal', featureidkey='properties.COD_POSTAL',
    color='temp_media', animation_frame='anio',
    color_continuous_scale='RdBu_r',
    hover_name='nombre_postal',
    hover_data={'temp_media': ':.2f', 'anio': True},
    labels={'temp_media': 'Temp media (°C)'},
    title='Temperatura media Meteocat por codigo postal',
    range_color=[temp_min_global, temp_max_global],
    **MAPA_BASE
)
fig_temp.update_layout(**LAYOUT_CLARO,
                       coloraxis_colorbar=colorbar_claro('Temp media (°C)'))
fig_temp.show()

### <font color='#C0392B'><b>5.5 Tests estadísticos Kruskal-Wallis y Mann-Whitney</b></font>


In [ ]:
VARS_KW = ['hora', 'dia_semana', 'mes', 'anio']

print(f"{'variable':<15} {'H_stat':>12} {'p_value':>14} {'sig':>5}")
print('-' * 50)

for var in VARS_KW:
    grupos = [
        df.filter(pl.col(var) == v)['mwh_total'].drop_nulls().to_numpy()
        for v in sorted(df[var].drop_nulls().unique().to_list())
    ]
    h_stat, p_val = kruskal(*grupos)
    sig = '✓' if p_val < 0.05 else '✗'
    print(f"{var:<15} {h_stat:>12.3f} {p_val:>14.4e} {sig:>5}")

print(f"\n{'variable':<15} {'mediana_0':>12} {'mediana_1':>12} {'p_value':>14} {'sig':>5}")
print('-' * 60)

for var in ['es_finde', 'es_festivo']:
    g0 = df.filter(pl.col(var) == 0)['mwh_total'].drop_nulls().to_numpy()
    g1 = df.filter(pl.col(var) == 1)['mwh_total'].drop_nulls().to_numpy()
    stat, p_val = mannwhitneyu(g0, g1, alternative='two-sided')
    sig = '✓' if p_val < 0.05 else '✗'
    print(f"{var:<15} {np.median(g0):>12.1f} {np.median(g1):>12.1f} {p_val:>14.4e} {sig:>5}")

grupos_cp = [
    df.filter(pl.col('cod_postal') == cp)['mwh_total'].drop_nulls().to_numpy()
    for cp in sorted(df['cod_postal'].unique().to_list())
]
h_stat, p_val = kruskal(*grupos_cp)
print(f"\n{'variable':<15} {'H_stat':>12} {'p_value':>14} {'sig':>5}")
print('-' * 50)
print(f"{'cod_postal':<15} {h_stat:>12.3f} {p_val:>14.4e} {'✓' if p_val < 0.05 else '✗':>5}")

Tests de hipótesis para las variables categóricas (Kruskal-Wallis y Mann-Whitney). Todas las variables son estadísticamente significativas (p < 0.05). cod_postal es el factor más discriminante del dataset (H_stat=279.152), por encima de cualquier variable temporal. hora es la variable temporal con mayor poder discriminante (H_stat=48.958), seguida de mes (6.355) y dia_semana (5.946).

---
# <font color='#1B3A5C'>  **6. Análisis de Correlación Spearman** </font>

In [ ]:
VARS_CORR = ['mwh_total', 'temp_mean', 'temp_max', 'temp_min',
             'humedad_mean', 'viento_mean', 'precipitacion_sum',
             'irradiancia_mean', 'lst_celsius']

df_corr = df.select(VARS_CORR).drop_nulls().to_pandas()
corr_matrix = df_corr.corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    ax=ax,
    linewidths=0.5
)
ax.set_title('Correlación de Spearman — variables numéricas', fontweight='bold')
plt.tight_layout()
plt.show()

Correlación de Spearman entre las variables numéricas. La correlación de mwh_total con las variables climáticas es baja pero significativa: temp_mean (0.16), irradiancia_mean (0.16) y lst_celsius (0.07); la relación en U con la temperatura explica el valor bajo, ya que Spearman subestima las relaciones no lineales. temp_mean, temp_max y temp_min correlacionan entre sí a 0.95-0.99, una colinealidad extrema, por lo que solo temp_mean entra al modelo y las otras dos se descartan. lst_celsius correlaciona con temp_mean a 0.87 pero no es redundante: la LST mide la temperatura superficial con variación espacial por barrio (UHI), mientras que temp_mean es igual para todos los CPs de la misma estación Meteocat. humedad_mean e irradiancia_mean aportan dimensiones independientes y entran al modelo. viento_mean y precipitacion_sum muestran una correlación cercana a 0 con mwh_total, y su aportación se revisará vía SHAP tras el modelado.

---
# <font color='#1B3A5C'>  **7. PCA** </font>

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

VARS_NUM = [
    'temp_mean',
    'humedad_mean', 'viento_mean', 'precipitacion_sum',
    'irradiancia_mean', 'lst_celsius'
]

df_pca_raw = df.select(VARS_NUM).drop_nulls().to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca_raw)

pca = PCA(n_components=len(VARS_NUM))
pca.fit(X_scaled)

var_exp_ratio = pca.explained_variance_ratio_
var_acum      = np.cumsum(var_exp_ratio)

print(f"{'PC':<6} {'Var explicada':>15} {'Var acumulada':>15}")
print('-' * 40)
for i, (v, va) in enumerate(zip(var_exp_ratio, var_acum)):
    print(f"PC{i+1:<4} {v:>14.4f} {va:>14.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for i, v in enumerate(var_exp_ratio):
    ax.bar(i + 1, v, color=C1, alpha=0.15, width=0.6, zorder=1)

ax.plot(range(1, len(var_acum) + 1), var_acum,
        color=C1, linewidth=2.5, zorder=3)
ax.scatter(range(1, len(var_acum) + 1), var_acum,
           color=C1, s=80, zorder=4,
           edgecolors='white', linewidths=1.5)

for i, v in enumerate(var_acum):
    ax.text(i + 1, v + 0.02, f'{v*100:.1f}%',
            ha='center', fontsize=8,
            color=C1, fontweight='bold')

ax.axhline(0.90, color='gray', linewidth=1,
           linestyle='--', alpha=0.6)
ax.text(len(var_acum) * 0.65, 0.91, '90% varianza',
        fontsize=8, color='gray')

ax.axvline(4, color=C3, linewidth=1,
           linestyle='--', alpha=0.6)
ax.text(4.1, 0.3, '4 PCs = 89.6%',
        fontsize=8, color=C3, fontweight='bold')

ax.set_title('Grafico de Codo — Varianza Acumulada PCA',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Componente Principal', fontsize=9)
ax.set_ylabel('Varianza Explicada Acumulada', fontsize=9)
ax.set_ylim(0, 1.08)
ax.set_xticks(range(1, len(var_acum) + 1))
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x*100:.0f}%')
)

plt.tight_layout()
plt.show()

In [ ]:
loadings_df = pd.DataFrame(
    pca.components_.T,
    index=VARS_NUM,
    columns=[f'PC{i+1}' for i in range(len(VARS_NUM))]
).round(4)

print("Loadings:")
print(loadings_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    loadings_df.iloc[:, :4],
    annot=True, fmt='.2f',
    cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=2, linecolor='white',
    annot_kws={'size': 10, 'fontweight': 'bold'},
    ax=ax,
    cbar_kws={'label': 'Loading', 'shrink': 0.8}
)

ax.set_title('Loadings PCA — PC1 a PC4',
             fontweight='bold', fontsize=12, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(fontsize=10, fontweight='bold')
plt.yticks(fontsize=9, rotation=0)
plt.tight_layout()
plt.show()

PCA de las variables meteorológicas. Cuatro componentes acumulan el 89.6% de la varianza, lo que confirma que las 6 variables aportan información diferenciada sin una redundancia problemática. PC1 (36.2%) es el eje térmico-solar: temp_mean (0.62), lst_celsius (0.57) e irradiancia_mean (0.47), que agrupa los días cálidos y soleados. PC2 (23.6%) es el eje de tiempo adverso: humedad alta (0.58) frente a viento fuerte (-0.57) e irradiancia baja (-0.37), que captura las condiciones meteorológicas desfavorables. PC3 (16.9%) es el eje de precipitación: precipitacion_sum domina con un loading de 0.94, una dimensión completamente independiente del resto. PC4 (12.8%) es el eje de viento húmedo: viento (0.73) y humedad (0.55) como combinación de condiciones de brisa marítima. El PCA es confirmatorio, no transformador: XGBoost y LSTM gestionan la colinealidad de forma nativa y reducir a componentes eliminaría la interpretabilidad de SHAP, por lo que las variables originales entran al modelo.

---
# <font color='#1B3A5C'>  **8. Conclusiones Globales** </font>

Dataset. El conjunto tiene 424.368 registros y 30 columnas (2019-2025, bloques de 6h, 42 códigos postales). Tras la limpieza no quedan nulos en mwh_total ni duplicados en datetime y cod_postal.

Calidad de datos. Se imputaron con mediana histórica los 220 bloques faltantes (08011 en agosto de 2025 y el 19-ago global) y los 378 registros con mwh_total = 0 en 3 fechas de fallo de OpenData BCN. Los 520 outliers erróneos en 6 CPs (acumulaciones de reporte) se imputaron con la mediana 2022-2024. La estación AN se reasignó a X2 con datos reales de Meteocat. Los nulos meteorológicos restantes (viento, irradiancia y temperatura/humedad de X2 en 2024-2025) son estructurales y se gestionan en feature engineering.

Estructura temporal de mwh_total. En el plano intradiario hay un pico a las 12h (120k MWh) y un valle a las 00h (71k MWh), una diferencia del 69%, y es la variable más discriminante del grupo temporal (Kruskal-Wallis H=48.958). En el plano semanal, los laborables rondan los 106k MWh frente a los 86k del domingo (aproximadamente un 19% menos), y los festivos reducen un 15% adicional respecto a los laborables. En el plano anual hay un doble pico verano-invierno con un valle en abril-mayo, una caída COVID en 2020 (en torno al 12%) y una recuperación progresiva hasta 2023. Todos los tests Kruskal-Wallis son significativos (p < 0.001).

Heterogeneidad espacial. El rango es de 8 veces entre barrios: desde 29.7k MWh (08033 Vallbona) hasta 233.6k MWh (08040 Zona Franca). cod_postal es el factor más discriminante del dataset (Kruskal-Wallis H=279.152, por encima de cualquier variable temporal). Los barrios industriales (08040, 08038, 08030) lideran el consumo con un CV bajo (en torno al 22-28%) y son los más predecibles, mientras que los barrios de servicios (08011, 08036, 08007-09) presentan un CV superior al 50% y son los más difíciles de predecir, donde el modelo concentrará los errores mayores.

Variables predictivas. Tienen señal alta hora, dia_semana, mes, es_finde, es_festivo y cod_postal, confirmadas por Kruskal-Wallis y Mann-Whitney (p < 0.001). Tienen señal moderada temp_mean, lst_celsius, humedad_mean e irradiancia_mean, significativas por Mann-Whitney con una relación no lineal (en J/U) con mwh_total. Tienen señal baja viento_mean y precipitacion_sum, significativas estadísticamente pero con una diferencia práctica mínima, cuya aportación se revisará vía SHAP. Se excluyen por data leakage mwh_industria, mwh_residencial y mwh_servicios, derivadas directamente del target. lst_celsius es complementaria a temp_mean: captura la variación espacial intraurbana por Urban Heat Island, con un Spearman de 0.87 entre ambas pero con señales distintas por naturaleza (aire frente a superficie).

Estacionariedad. ADF confirma la estacionariedad en media (p < 0.05) en los 5 casos analizados (global más 4 CPs representativos). KPSS rechaza la estacionariedad en varianza, con cambios estructurales visibles en el STL (COVID 2020, recuperación gradual, mayor variabilidad industrial en 2024-2025). XGBoost y LSTM/GRU son robustos a esta condición; SARIMA requerirá diferenciación estacional para cumplir sus supuestos formales.

Lags identificados. lag_4 (24h, PACF 0.75) es la contribución directa más fuerte de la serie. lag_28 (7 días, ACF 0.87) es el más parecido de toda la serie y captura el patrón semanal. lag_1 (6h, PACF 0.37) es la contribución directa moderada del bloque inmediato anterior. rolling_mean_7d captura la tendencia reciente sin depender de un solo bloque puntual. lag_2 y lag_3 (12h, 18h) y rolling_std_7d se incorporan para los modelos de ML (árboles/LSTM): lag_2 y lag_3 completan el día anterior (PACF lag_3 = 0.47) y rolling_std_7d aporta la volatilidad para la alerta temprana.

PCA confirmatorio. Cuatro componentes explican el 89.6% de la varianza meteorológica: PC1 es el eje térmico-solar (temp + LST + irradiancia), PC2 el de tiempo adverso, PC3 el de precipitación y PC4 el de viento húmedo. Las variables originales entran al modelo para mantener la interpretabilidad SHAP, ya que el PCA es confirmatorio, no transformador.

Tendencia térmica (2019-2025). temp_mean sube +0.31 °C/año y humedad_mean desciende -0.31%/año de forma simétrica, indicativo de un calentamiento urbano progresivo en el período analizado. Con 7 puntos anuales la tendencia es indicativa, no concluyente estadísticamente, y se documenta como contexto para la interpretación del modelo.

Limitaciones documentadas. MODIS MOD11A1 (Terra, en torno a las 10:30h local) asigna el mismo valor de LST a los 4 bloques del día; MYD11A1 (Aqua, en torno a las 13:30h) capturaría mejor el pico UHI, y se documenta como línea futura. La cobertura de LST en invierno es limitada: febrero-marzo tienen solo un 32-34% de registros válidos por la nubosidad, y los valores disponibles corresponden a días despejados y fríos, lo que infraestima la LST media real. Los CPs 08010 Pl. Catalunya y 08008-08009 Dreta de l'Eixample tienen un consumo bajo por la superficie reducida del CP, no por una baja actividad económica. En el CP 08037 se imputaron unos 475 bloques (jul-nov 2025) con mediana histórica por ser valores imposibles (6-7 veces lo normal): casi medio 2025 de ese CP es sintético, por lo que sus resultados de 2025 se comparan contra la mediana histórica, no contra la demanda real.

---
# <font color='#1B3A5C'>  **9. Feature Engineering** </font>

Lags temporales. lag_1 (6h antes) tiene una contribución directa confirmada por la PACF (0.37). lag_2 y lag_3 (12h y 18h antes) completan el bloque del día anterior, con PACF lag_3 = 0.47 (contribución directa fuerte) y lag_2 = -0.22, y aportan la forma intradía completa a XGBoost/LightGBM/LSTM. lag_4 (24h antes) es el más importante, con PACF 0.75, el mismo bloque del día anterior. lag_28 (7 días antes) tiene ACF 0.87 y captura el patrón semanal completo. rolling_mean_7d es el promedio de los últimos 7 días y captura la tendencia reciente. rolling_std_7d es la desviación estándar de los últimos 7 días y captura la volatilidad reciente, útil para el componente de alerta temprana (detección de picos y anomalías). lag_56 y lag_84 (2 y 3 semanas) tienen correlación pero decrece, y como lag_28 ya captura el patrón semanal, se evalúan tras el modelado vía SHAP.

Variables de interacción. hora_x_finde = hora * es_finde captura que el perfil horario del domingo es cualitativamente distinto al del miércoles, solo para XGBoost y LightGBM. is_covid = 1 si anio == 2020 controla el cambio estructural por la pandemia.

Variables climáticas derivadas. HDD = max(0, 15 - temp_mean) son los grados de calefacción [Kuru & Calis 2019]. CDD = max(0, temp_mean - 22) son los grados de refrigeración [Kuru & Calis 2019]. lst_nublado = 1 si lst_celsius es nulo es un flag de fiabilidad de MODIS. precipitacion_llueve = 1 si precipitacion_sum > 0 es la binarización confirmada en el EDA (ambas medianas en 0, con información útil solo en si llueve o no).

Imputación meteorológica de X2. La temperatura de X2 en 2025 se imputa desde X4 con un factor de corrección histórico (ratio X2/X4, 2019-2023). El viento y la irradiancia de X2 siempre se toman de X4 (X2 nunca tuvo sensores). lst_celsius sin cobertura se respalda espacialmente con MODIS. En irradiancia_mean se topan a 0 los valores negativos de sensor.

Vecinos geográficos (despriorizado). lag1_vecinos es el consumo promedio de los CPs colindantes en t-1, calculado con shapely sobre BARCELONA.geojson. Se descarta si hay restricción de tiempo y se documenta como línea futura.

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

registros = df.to_dicts()

db['dataset_clean'].drop()
db['dataset_clean'].insert_many(registros)

print(f"dataset_clean guardado: {db['dataset_clean'].count_documents({}):,} registros")
print(f"Columnas: {df.columns}")

In [ ]:
doc_test = db['dataset_clean'].find_one({}, {'_id': 0})
print("Ejemplo de registro:")
for k, v in doc_test.items():
    print(f"  {k:<25} {v}")

In [ ]:
elapsed = time.time() - start_time
print(f"Tiempo de ejecución del notebook: {elapsed/60:.1f} min ({elapsed:.0f} seg)")